# Probe-based single-cell RNA-seq (ProBac-seq)

Figures 2A, 2B, 2C, S5, S6, S7, S8, S9A, S9B

Environment: `single_cell_R_env` (`envs/single_cell_R_env.yml`)

Inputs:
- `*_raw_feature_bc_matrix.h5` for CTRL/INF 15, 30, 45 min (GSE349090), in `data_dir/GSE349090/`
- `INF180min_raw_feature_bc_matrix.h5` (GSE349086), in `data_dir/GSE349086/`
- `bulk_RNAseq_htseq_raw_counts.csv` (GSE348942), in `data_dir/GSE348942/`

In [ ]:
suppressPackageStartupMessages({
  library(yaml)
  library(Matrix)
  library(data.table)
  library(Seurat)
  library(ggplot2)
  library(ggrepel)
})

`%||%` <- function(a, b) if (is.null(a)) b else a

ROOT <- ".."
config <- read_yaml(file.path(ROOT, "config.yaml"))
DATA_DIR <- config$data_dir
OUT_DIR  <- config$out_dir
FIG_DIR  <- file.path(OUT_DIR, "figures")
SRC_DIR  <- file.path(OUT_DIR, "source_data")
SC_DIR   <- file.path(OUT_DIR, "scRNAseq")
for (d in c(FIG_DIR, SRC_DIR, SC_DIR)) dir.create(d, recursive = TRUE, showWarnings = FALSE)

FIG_FORMATS  <- c("png", "pdf")
SAVE_OUTPUTS <- FALSE

options(repr.plot.width = 9, repr.plot.height = 6)
options(datatable.print.class = FALSE)

## Configuration and parameters

In [ ]:
RUN_SAMPLES <- c("Ctrl_45m", "Inf_45m")

h5 <- function(gse, lib) vapply(lib, function(l) {
  f <- Sys.glob(file.path(DATA_DIR, gse, paste0("*", l, "_raw_feature_bc_matrix.h5")))
  if (length(f) != 1) stop("expected one h5 file for ", l, " in ", file.path(DATA_DIR, gse))
  f
}, character(1), USE.NAMES = FALSE)

SAMPLES_exp1 <- data.table(
  sample_id = c("Ctrl_15m", "Inf_15m", "Ctrl_30m", "Inf_30m", "Ctrl_45m", "Inf_45m"),
  timepoint = c(15, 15, 30, 30, 45, 45),
  condition = c("control", "infected", "control", "infected", "control", "infected"),
  path      = h5("GSE349090", c("CTRL15min", "INF15min", "CTRL30min", "INF30min", "CTRL45min", "INF45min")),
  type      = "h5"
)

SAMPLES_inf180 <- data.table(
  sample_id = "INF_180min", timepoint = 180, condition = "infected",
  path = h5("GSE349086", "INF180min"), type = "h5"
)

BULK_COUNTS <- file.path(DATA_DIR, "GSE348942", "bulk_RNAseq_htseq_raw_counts.csv")

stopifnot(file.exists(SAMPLES_exp1$path), file.exists(SAMPLES_inf180$path), file.exists(BULK_COUNTS))

params_base <- list(

  level      = "gene",   # "probe" | "gene"
  agg_method = "sum",    # "sum" | "max" | "median"   (used only if level=="gene")

  out_root     = "results",
  save_outputs = SAVE_OUTPUTS,

  h5_use_names       = TRUE,   # TRUE -> /matrix/features/name, FALSE -> .../id
  load_min_total_umi = 3,      # memory measure: drops empty whitelist barcodes

  phage_prefix      = "^MS2",
  phage_ignore_case = TRUE,
  protect_regex     = "^LELOBEKK_[0-9]+$",  # bare locus tags: do NOT strip
  probe_suffix      = "[-_][0-9]+$",        # the probe index that gets stripped

  qc_feat_basis          = "gene",   # "gene" | "probe"
  qc_umi_basis           = "gene",   # "gene" | "probe"
  qc_phage_basis         = "probe",  # "probe" -- do not change
  qc_basis               = "probe",  # legacy fallback for the three above
  prefilter_min_host_umi = 100,
  qc_scope               = "sample",
  umi_lower_q  = 0.01, umi_upper_q  = 0.99,
  min_host_umi = 100,  max_host_umi = Inf,
  feat_lower_q = 0.01, feat_upper_q = 0.99,
  min_host_features = 50, max_host_features = Inf,
  min_cells_per_feature = 3,          # phage features are always exempt

  valid_probe_regex = "rev",
  min_valid_probes  = 7,
  min_phage_umi     = 1,
  n_load_grades     = 4,
  valley_adjust     = 1,
  self_q_infected   = 0.90,
  manual_thr_infected = NULL, manual_thr_ambiguous = NULL,
  phage_q_ambiguous = 0.95, phage_q_infected = 0.99,

  norm_basis         = "host",   # host-only size factors. Do not change.
  scale_factor       = NULL,     # NULL -> median host total
  scale_factor_scope = "global",

  hvg_include_phage = FALSE,     # phage must not drive the embedding
  n_hvg    = 1000,
  npcs     = 20,
  dims_use = 1:10,
  resolution = 0.2,
  vars_to_regress = NULL,
  integrate = "none", harmony_group = "sample_id",

  marker_only_pos     = TRUE,
  marker_min_pct      = 0.10,
  marker_logfc_thresh = 0.25,
  marker_test         = "wilcox",

  analysis_scope         = "per_sample",
  min_cells_per_analysis = 100,
  analysis_samples       = NULL,   # NULL = every sample that was read
  include_controls_for_cutoff = TRUE,

  outlier_genes = NULL,

  seed = 1234
)

params_exp1 <- modifyList(params_base, list(
  sample_sheet = SAMPLES_exp1,
  out_root     = file.path(SC_DIR, "exp1"),
  level        = "gene",
  agg_method   = "max",
  phage_metric       = "percent",
  phage_method       = "control_quantile",
  phage_cutoff_scope = "global",
  min_percent_ambiguous = 0.05,   # in percent, because phage_metric = "percent"
  min_percent_infected  = 0.5,
  analysis_samples = if (identical(RUN_SAMPLES, "all")) NULL else RUN_SAMPLES,
  include_controls_for_cutoff = TRUE
))

params_inf180 <- modifyList(params_base, list(
  sample_sheet = SAMPLES_inf180,
  out_root     = file.path(SC_DIR, "inf180"),
  level        = "gene",
  agg_method   = "max",
  phage_metric       = "per_host",
  phage_method       = "otsu",
  phage_cutoff_scope = "per_sample",
  min_percent_ambiguous = 0.005,  # ratio ~ 0.5% of counts
  min_percent_infected  = 0.05,   # ratio ~ 4.8% of counts
  analysis_samples = "INF_180min",
  include_controls_for_cutoff = FALSE
))

make_run_tag <- function(p)
  if (identical(p$level, "probe")) "probe" else paste0("gene_", p$agg_method)

## Functions

In [ ]:
read_sample_sheet <- function(x) {
  ss <- if (is.character(x) && length(x) == 1) fread(x) else as.data.table(copy(x))
  req <- c("sample_id", "timepoint", "condition", "path")
  miss <- setdiff(req, names(ss))
  if (length(miss)) stop("sample sheet missing columns: ", paste(miss, collapse = ", "))
  if (!"type" %in% names(ss)) ss[, type := "auto"]
  ss[, condition := tolower(condition)]
  bad <- setdiff(unique(ss$condition), c("infected", "control"))
  if (length(bad)) stop("condition must be 'infected' or 'control'; got: ",
                        paste(bad, collapse = ", "))
  if (anyDuplicated(ss$sample_id)) stop("sample_id values must be unique")
  ss[]
}

resolve_load_set <- function(ss, p) {
  sel <- p$analysis_samples
  if (is.null(sel) || identical(sel, "all")) return(ss[])

  miss <- setdiff(sel, ss$sample_id)
  if (length(miss))
    stop("analysis_samples not in the sheet: ", paste(miss, collapse = ", "),
         "\n  available: ", paste(ss$sample_id, collapse = ", "))

  load_ids <- sel
  needs_ctrl <- identical(p$phage_method, "control_quantile") ||
                identical(p$phage_method, "auto")
  if (isTRUE(p$include_controls_for_cutoff) && needs_ctrl) {
    ctrl <- ss[condition == "control", sample_id]
    if (identical(p$phage_cutoff_scope, "per_timepoint"))
      ctrl <- ss[condition == "control" &
                   timepoint %in% ss[sample_id %in% sel, timepoint], sample_id]
    extra <- setdiff(ctrl, load_ids)
    if (length(extra)) {
      message("  + reading control sample(s) for the infection cutoff only: ",
              paste(extra, collapse = ", "))
      load_ids <- c(load_ids, extra)
    }
  }
  out <- ss[sample_id %in% load_ids]
  if (identical(p$phage_method, "control_quantile") &&
      !any(out$condition == "control"))
    warning("phage_method='control_quantile' but no control sample is being read. ",
            "Classification will fail; set include_controls_for_cutoff=TRUE, add a ",
            "control to analysis_samples, or use phage_method='auto'/'otsu'.")
  out[]
}

read_one_matrix <- function(path, type = "h5", p) {
  m <- Seurat::Read10X_h5(path, use.names = p$h5_use_names, unique.features = TRUE)
  if (is.list(m)) {
    message("   feature types present: ", paste(names(m), collapse = ", "))
    m <- do.call(rbind, m)
  }
  m <- as(as(m, "CsparseMatrix"), "dgCMatrix")
  if (is.null(colnames(m))) colnames(m) <- paste0("cell", seq_len(ncol(m)))
  m
}

is_phage <- function(feats, p)
  grepl(p$phage_prefix, feats, ignore.case = isTRUE(p$phage_ignore_case))

probe_to_gene <- function(probe_names, p) {
  g <- probe_names
  to_modify <- !grepl(p$protect_regex, probe_names)
  g[to_modify] <- gsub(p$probe_suffix, "", probe_names[to_modify])
  g
}

aggregate_probes <- function(raw_matrix, method = "sum", p) {
  if (tolower(method) == "none") return(raw_matrix)
  stopifnot(method %in% c("sum", "max", "median"))

  raw_matrix <- as(as(raw_matrix, "CsparseMatrix"), "dgCMatrix")
  dt <- as.data.table(Matrix::summary(raw_matrix))
  probe_names <- rownames(raw_matrix)
  gene_names  <- probe_to_gene(probe_names, p)
  dt[, gene := gene_names[i]]

  dt_agg <- switch(method,
    "sum"    = dt[, .(x = sum(x)),    by = .(gene, j)],
    "max"    = dt[, .(x = max(x)),    by = .(gene, j)],
    "median" = dt[, .(x = median(x)), by = .(gene, j)]
  )
  dt_agg <- dt_agg[x != 0]

  unique_genes <- unique(gene_names)
  gene_to_i <- setNames(seq_along(unique_genes), unique_genes)

  sparseMatrix(
    i = unname(gene_to_i[dt_agg$gene]), j = dt_agg$j, x = dt_agg$x,
    dims = c(length(unique_genes), ncol(raw_matrix)),
    dimnames = list(unique_genes, colnames(raw_matrix))
  )
}

cbind_fill_union <- function(mat_list) {
  all_feats <- Reduce(union, lapply(mat_list, rownames))
  padded <- lapply(mat_list, function(m) {
    missing <- setdiff(all_feats, rownames(m))
    if (length(missing)) {
      pad <- sparseMatrix(i = integer(0), j = integer(0), x = numeric(0),
                          dims = c(length(missing), ncol(m)),
                          dimnames = list(missing, colnames(m)))
      m <- rbind(m, pad)
    }
    m[all_feats, , drop = FALSE]
  })
  do.call(cbind, padded)
}

load_all_samples <- function(sample_sheet, p) {
  mats <- vector("list", nrow(sample_sheet)); metas <- mats
  load_log <- vector("list", nrow(sample_sheet))

  for (k in seq_len(nrow(sample_sheet))) {
    row <- sample_sheet[k]
    message("Reading ", row$sample_id, " ...")
    m <- read_one_matrix(row$path, row$type, p)
    n_raw <- ncol(m)
    if (p$load_min_total_umi > 0) {
      tot <- Matrix::colSums(m)
      m <- m[, tot >= p$load_min_total_umi, drop = FALSE]
    }
    colnames(m) <- paste0(row$sample_id, "|", colnames(m))

    mats[[k]] <- m
    metas[[k]] <- data.table(cell = colnames(m), sample_id = row$sample_id,
                             timepoint = row$timepoint, condition = row$condition)
    load_log[[k]] <- data.table(sample_id = row$sample_id,
                                n_barcodes_in_file = n_raw,
                                n_barcodes_loaded  = ncol(m),
                                n_features = nrow(m))
    message("   ", nrow(m), " features | ", format(n_raw, big.mark = ","),
            " barcodes in file -> ", format(ncol(m), big.mark = ","),
            " kept at >=", p$load_min_total_umi, " UMI")
    rm(m); gc(verbose = FALSE)
  }

  mat  <- cbind_fill_union(mats)
  rm(mats); gc(verbose = FALSE)
  meta <- rbindlist(metas); setkey(meta, cell); meta <- meta[colnames(mat)]

  ll <- rbindlist(load_log)
  print(ll)
  message("combined: ", nrow(mat), " features x ",
          format(ncol(mat), big.mark = ","), " barcodes")
  list(counts_probe = mat, meta = meta, load_log = ll)
}

diagnose_feature_names <- function(feats, p) {
  is_ph <- is_phage(feats, p)
  gn <- probe_to_gene(feats, p)
  n_host_probes <- sum(!is_ph); n_host_genes <- length(unique(gn[!is_ph]))
  n_phage_probes <- sum(is_ph); n_phage_genes <- length(unique(gn[is_ph]))

  cat("\n--- feature names ---\n")
  cat("  host  : ", n_host_probes, " probes -> ", n_host_genes, " genes\n", sep = "")
  cat("  phage : ", n_phage_probes, " probes -> ", n_phage_genes, " genes  (",
      paste(head(feats[is_ph], 4), collapse = ", "), ", ...)\n", sep = "")
  cat("  protected by regex: ", sum(grepl(p$protect_regex, feats)), "\n", sep = "")
  cat("  host probes per gene:\n"); print(table(table(gn[!is_ph])))

  if (n_phage_probes == 0)
    warning("No features match phage_prefix '", p$phage_prefix, "'.")
  if (n_host_genes == n_host_probes)
    warning("NO host probes collapse -- level='gene' equals level='probe' here.")
  invisible(list(n_host_probes = n_host_probes, n_host_genes = n_host_genes,
                 n_phage_probes = n_phage_probes, n_phage_genes = n_phage_genes))
}

resolve_outliers <- function(p) as.character(p$outlier_genes %||% character(0))

drop_outlier_features <- function(mat, p, verbose = TRUE) {
  og <- p$outlier_genes
  if (is.null(og) || !length(og)) return(mat)
  gn  <- probe_to_gene(rownames(mat), p)
  hit <- tolower(gn) %in% tolower(og) & !is_phage(rownames(mat), p)
  unmatched <- setdiff(tolower(og), tolower(gn))
  if (verbose) {
    message("  outlier genes: dropping ", sum(hit), " feature(s) = ",
            length(unique(gn[hit])), " of ", length(og), " listed genes")
    if (length(unmatched))
      message("    not present in this panel (", length(unmatched), "): ",
              paste(head(og[tolower(og) %in% unmatched], 8), collapse = ", "),
              if (length(unmatched) > 8) ", ..." else "")
  }
  if (!any(hit)) return(mat)
  mat[!hit, , drop = FALSE]
}

In [ ]:
compute_qc_metrics <- function(mat_umi, meta, p, mat_feat = NULL, mat_phage = NULL) {
  mat_feat  <- mat_feat  %||% mat_umi
  mat_phage <- mat_phage %||% mat_umi

  cells <- colnames(mat_umi)
  if (!identical(cells, colnames(mat_feat))) {
    if (!setequal(cells, colnames(mat_feat)))
      stop("mat_feat does not cover the same barcodes as mat_umi")
    mat_feat <- mat_feat[, cells, drop = FALSE]
  }
  if (!identical(cells, colnames(mat_phage))) {
    if (!setequal(cells, colnames(mat_phage)))
      stop("mat_phage does not cover the same barcodes as mat_umi")
    mat_phage <- mat_phage[, cells, drop = FALSE]
  }

  ph_phage <- is_phage(rownames(mat_phage), p)
  host_u <- mat_umi  [!is_phage(rownames(mat_umi),  p), , drop = FALSE]
  host_f <- mat_feat [!is_phage(rownames(mat_feat), p), , drop = FALSE]
  host_p <- mat_phage[!ph_phage, , drop = FALSE]
  phg    <- mat_phage[ ph_phage, , drop = FALSE]

  qc <- data.table(
    cell       = cells,
    host_umi   = Matrix::colSums(host_u),
    host_feat  = Matrix::colSums(host_f != 0),
    host_umi_probe = Matrix::colSums(host_p),
    phage_umi  = if (sum(ph_phage)) Matrix::colSums(phg)      else 0,
    phage_feat = if (sum(ph_phage)) Matrix::colSums(phg != 0) else 0
  )
  qc[, total_umi   := host_umi_probe + phage_umi]
  qc[, phage_ratio := fifelse(total_umi > 0, phage_umi / total_umi, 0)]
  qc[, percent_MS2 := 100 * phage_ratio]
  qc[, phage_per_1k := fifelse(host_umi_probe > 0,
                               1000 * phage_umi / host_umi_probe, NA_real_)]
  qc[, phage_per_host := fifelse(host_umi_probe > 0,
                                 phage_umi / host_umi_probe, NA_real_)]

  setkey(qc, cell); setkey(meta, cell)
  meta[qc][]
}

compute_valid_viral_probes <- function(counts_probe, p) {
  ph <- rownames(counts_probe)[is_phage(rownames(counts_probe), p)]
  if (!length(ph)) stop("no phage features found; cannot compute Valid_Viral_Probes")

  sel <- ph
  if (!is.null(p$valid_probe_regex)) {
    sel <- grep(p$valid_probe_regex, ph, value = TRUE,
                ignore.case = isTRUE(p$phage_ignore_case))
    if (!length(sel)) {
      warning("valid_probe_regex '", p$valid_probe_regex, "' matched none of the ",
              length(ph), " phage probes. Falling back to ALL phage probes.")
      sel <- ph
    }
  }
  message("  Valid_Viral_Probes: ", length(sel), " of ", length(ph),
          " phage probes; threshold >= ", p$min_valid_probes)

  n <- Matrix::colSums(counts_probe[sel, , drop = FALSE] > 0)
  data.table(cell = colnames(counts_probe),
             n_valid_probes = as.integer(n),
             valid_viral_probes = n >= p$min_valid_probes)
}

compute_qc_thresholds <- function(qc, p) {
  qc <- copy(qc)
  qc[, passes_prefilter := host_umi >= p$prefilter_min_host_umi]

  .lo <- function(v, q, floor_val) {
    if (is.null(q) || is.na(q) || q <= 0) return(as.numeric(floor_val))
    max(as.numeric(quantile(v, q, na.rm = TRUE)), as.numeric(floor_val))
  }
  .hi <- function(v, q, ceil_val) {
    if (is.null(q) || is.na(q) || q >= 1) return(as.numeric(ceil_val))
    min(as.numeric(quantile(v, q, na.rm = TRUE)), as.numeric(ceil_val))
  }

  calc <- function(d) {
    dd <- d[passes_prefilter == TRUE]
    if (nrow(dd) < 20)
      warning("Fewer than 20 barcodes clear the prefilter in a group; ",
              "quantile thresholds will be unstable.")
    q_umi_lo  <- if (is.null(p$umi_lower_q)  || p$umi_lower_q  <= 0) NA_real_ else
      as.numeric(quantile(dd$host_umi,  p$umi_lower_q,  na.rm = TRUE))
    q_feat_lo <- if (is.null(p$feat_lower_q) || p$feat_lower_q <= 0) NA_real_ else
      as.numeric(quantile(dd$host_feat, p$feat_lower_q, na.rm = TRUE))
    data.table(
      n_prefilter = nrow(dd),
      umi_lo  = .lo(dd$host_umi,  p$umi_lower_q,  p$min_host_umi),
      umi_hi  = .hi(dd$host_umi,  p$umi_upper_q,  p$max_host_umi),
      feat_lo = .lo(dd$host_feat, p$feat_lower_q, p$min_host_features),
      feat_hi = .hi(dd$host_feat, p$feat_upper_q, p$max_host_features),
      umi_lo_q = round(q_umi_lo, 1), feat_lo_q = round(q_feat_lo, 1),
      umi_lo_from  = if (is.na(q_umi_lo))  "floor" else
        if (q_umi_lo  > p$min_host_umi)      "quantile" else "floor",
      feat_lo_from = if (is.na(q_feat_lo)) "floor" else
        if (q_feat_lo > p$min_host_features) "quantile" else "floor")
  }

  if (identical(p$qc_scope, "sample")) qc[, calc(.SD), by = sample_id]
  else cbind(data.table(sample_id = "ALL"), calc(qc))
}

apply_qc_filter <- function(qc, thr, p) {
  qc <- copy(qc)
  if (identical(p$qc_scope, "sample")) {
    qc <- merge(qc, thr, by = "sample_id", all.x = TRUE)
  } else {
    for (cn in setdiff(names(thr), "sample_id")) qc[, (cn) := thr[[cn]]]
  }
  qc[, keep_umi  := host_umi  >= umi_lo  & host_umi  <= umi_hi]
  qc[, keep_feat := host_feat >= feat_lo & host_feat <= feat_hi]
  qc[, keep      := keep_umi & keep_feat]

  smry <- qc[, .(n_barcodes = .N,
                 n_pass_umi = sum(keep_umi), n_pass_feat = sum(keep_feat),
                 n_keep = sum(keep), pct_keep = round(100 * mean(keep), 1),
                 med_host_umi_kept  = as.numeric(median(host_umi[keep])),
                 med_host_feat_kept = as.numeric(median(host_feat[keep]))),
             by = .(sample_id, timepoint, condition)][order(timepoint, condition)]
  list(qc = qc, summary = smry)
}

filter_features <- function(mat, p) {
  n_cells <- Matrix::rowSums(mat != 0)
  is_ph   <- is_phage(rownames(mat), p)
  keep    <- is_ph | n_cells >= p$min_cells_per_feature
  message("  feature filter: keeping ", sum(keep), " / ", length(keep),
          " features (", sum(is_ph), " phage protected)")
  mat[keep, , drop = FALSE]
}

qc_with_offset_umi <- function(qc) {
  d <- data.table::copy(data.table::as.data.table(qc))
  if ("host_umi_probe" %in% names(d)) d[, host_umi := host_umi_probe]
  d[]
}

In [ ]:
.metric_col <- function(p) switch(p$phage_metric,
  "percent" = "percent_MS2", "ratio" = "phage_ratio",
  "per_host" = "phage_per_host", "per_1k_host" = "phage_per_1k",
  "count" = "phage_umi",
  stop("phage_metric must be 'percent', 'per_host', 'ratio', 'per_1k_host' or 'count'"))

assess_separability <- function(v, adjust = 1) {
  pos <- v[is.finite(v) & v > 0]
  out <- list(n_positive = length(pos), bimodal = FALSE, valley = NA_real_,
              valley_depth = NA_real_, mode_sep_log10 = NA_real_, note = "")
  if (length(pos) < 100) { out$note <- "too few positive cells"; return(out) }
  x <- log10(pos)
  if (diff(range(x)) < 0.05) { out$note <- "values span < 12%; nothing to split"
    return(out) }
  d  <- density(x, adjust = adjust, n = 2048)
  dy <- diff(d$y)
  pk <- which(c(FALSE, dy[-length(dy)] > 0 & dy[-1] <= 0))
  if (length(pk) < 2) { out$note <- "single mode -- no gap to cut at"; return(out) }
  two <- sort(pk[order(d$y[pk], decreasing = TRUE)][1:2])
  seg <- two[1]:two[2]
  vi  <- seg[which.min(d$y[seg])]
  out$valley         <- 10^d$x[vi]
  out$valley_depth   <- 1 - d$y[vi] / min(d$y[two[1]], d$y[two[2]])
  out$mode_sep_log10 <- abs(d$x[two[2]] - d$x[two[1]])
  out$bimodal <- out$valley_depth > 0.25 && out$mode_sep_log10 > 0.3
  out$note <- if (out$bimodal) "two separated modes" else
    sprintf("modes present but not separated (depth %.2f, %.2gx apart)",
            out$valley_depth, 10^out$mode_sep_log10)
  out
}

.otsu_threshold <- function(x, nbins = 256) {
  rng <- range(x); if (!is.finite(diff(rng)) || diff(rng) == 0) return(NA_real_)
  br  <- seq(rng[1], rng[2], length.out = nbins + 1)
  h   <- hist(x, breaks = br, plot = FALSE)$counts
  mid <- br[-1] - diff(br) / 2
  tot <- sum(h)
  w0  <- cumsum(h) / tot; w1 <- 1 - w0
  cs  <- cumsum(h * mid)
  mT  <- cs[length(cs)] / tot
  m0  <- cs / pmax(cumsum(h), 1e-12)
  m1  <- (mT * tot - cs) / pmax(tot - cumsum(h), 1e-12)
  bcv <- w0 * w1 * (m0 - m1)^2
  bcv[!is.finite(bcv)] <- -Inf
  mid[which.max(bcv)]
}

.density_valley <- function(x, adjust = 1) {
  if (length(x) < 50) return(NA_real_)
  d  <- density(x, adjust = adjust, n = 2048)
  dy <- diff(d$y)
  pk <- which(c(FALSE, dy[-length(dy)] > 0 & dy[-1] <= 0))
  if (length(pk) < 2) return(NA_real_)
  two <- sort(pk[order(d$y[pk], decreasing = TRUE)][1:2])
  seg <- two[1]:two[2]
  d$x[seg][which.min(d$y[seg])]
}

.mixture_split <- function(x) {
  if (!requireNamespace("mclust", quietly = TRUE)) return(NA_real_)
  if (length(x) < 100) return(NA_real_)
  fit <- try(mclust::Mclust(x, G = 2, modelNames = "V", verbose = FALSE), silent = TRUE)
  if (inherits(fit, "try-error") || is.null(fit)) return(NA_real_)
  o <- order(fit$parameters$mean); lo <- o[1]; hi <- o[2]
  g <- seq(min(x), max(x), length.out = 4000)
  dl <- fit$parameters$pro[lo] * dnorm(g, fit$parameters$mean[lo],
          sqrt(fit$parameters$variance$sigmasq[lo]))
  dh <- fit$parameters$pro[hi] * dnorm(g, fit$parameters$mean[hi],
          sqrt(fit$parameters$variance$sigmasq[hi]))
  g[which.min(abs(dl - dh))]
}

phage_cutoff_candidates <- function(v, p, valid = NULL) {
  pos <- v[is.finite(v) & v > 0]
  x   <- log10(pos)
  inv <- function(z) if (is.na(z)) NA_real_ else 10^z
  cand <- data.table(
    method = c("otsu", "valley", "mixture", "self_quantile", "manual"),
    thr = c(inv(.otsu_threshold(x)),
            inv(.density_valley(x, p$valley_adjust %||% 1)),
            inv(.mixture_split(x)),
            as.numeric(quantile(v, p$self_q_infected %||% 0.90, na.rm = TRUE)),
            if (is.null(p$manual_thr_infected)) NA_real_ else
              as.numeric(p$manual_thr_infected)))
  cand[, thr_after_floor := pmax(thr, p$min_percent_infected, na.rm = FALSE)]
  cand[, pct_cells_over := round(100 * sapply(thr_after_floor,
        function(t) if (is.na(t)) NA_real_ else mean(v > t)), 2)]
  if (!is.null(valid))
    cand[, pct_cells_infected := round(100 * sapply(thr_after_floor,
          function(t) if (is.na(t)) NA_real_ else mean(v > t & valid)), 2)]
  cand[, n_phage_positive := length(pos)]
  cand[]
}

.cut_distribution <- function(d, p, mth) {
  cand <- phage_cutoff_candidates(d$phage_metric_value, p, d$valid_viral_probes)
  raw  <- cand[method == mth, thr]          # `mth` is the argument, `method` the column
  used <- mth
  if (!length(raw) || is.na(raw)) {
    alt <- cand[!is.na(thr) & method != "self_quantile"][1]
    if (!nrow(alt)) alt <- cand[method == "self_quantile"]
    warning("phage_method='", mth, "' produced no cutoff; falling back to '",
            alt$method, "'.")
    raw <- alt$thr; used <- paste0(alt$method, " (fallback from ", mth, ")")
  }
  thr_amb <- p$manual_thr_ambiguous %||% p$min_percent_ambiguous
  thr_inf <- max(raw, p$min_percent_infected)
  data.table(method = used, n_control = 0L,
             q_ambiguous_raw = NA_real_, q_infected_raw = signif(raw, 4),
             thr_ambiguous = thr_amb, thr_infected = thr_inf,
             floor_binding_ambiguous = NA, floor_binding_infected = thr_inf > raw)
}

.cut_control_quantile <- function(d, p) {
  ctrl <- d[condition == "control"]
  if (!nrow(ctrl))
    stop("phage_method='control_quantile' but there are no control cells in this ",
         "group. Set include_controls_for_cutoff=TRUE, or use 'auto'/'otsu'.")
  if (nrow(ctrl) < 50)
    warning("Only ", nrow(ctrl), " control cells for the cutoff; the upper ",
            "quantiles will be very noisy.")
  q_amb <- as.numeric(quantile(ctrl$phage_metric_value, p$phage_q_ambiguous, na.rm = TRUE))
  q_inf <- as.numeric(quantile(ctrl$phage_metric_value, p$phage_q_infected,  na.rm = TRUE))
  thr_amb <- max(q_amb, p$min_percent_ambiguous)
  thr_inf <- max(q_inf, p$min_percent_infected)
  data.table(method = "control_quantile", n_control = nrow(ctrl),
             q_ambiguous_raw = signif(q_amb, 4), q_infected_raw = signif(q_inf, 4),
             thr_ambiguous = thr_amb, thr_infected = thr_inf,
             floor_binding_ambiguous = thr_amb > q_amb,
             floor_binding_infected  = thr_inf > q_inf)
}

add_infection_grade <- function(qc, p, n_bins = 4, verbose = TRUE) {
  qc <- copy(qc)
  v <- qc$phage_per_host
  qs <- unique(as.numeric(quantile(v, seq(0, 1, length.out = n_bins + 1), na.rm = TRUE)))
  if (length(qs) < 3) { qc[, phage_load_grade := factor("all")]; return(qc[]) }
  labs <- paste0("G", seq_len(length(qs) - 1))
  labs[1] <- paste0(labs[1], "_lowest")
  labs[length(labs)] <- paste0(labs[length(labs)], "_highest")
  qc[, phage_load_grade := cut(v, qs, include.lowest = TRUE, labels = labs)]
  if (verbose) {
    cat("\nphage load grades (quantiles of phage:host ratio):\n")
    print(qc[, .(n = .N, median_percent_MS2 = round(median(percent_MS2), 1)),
             by = phage_load_grade][order(phage_load_grade)])
  }
  qc[]
}

classify_infection <- function(qc_kept, p, verbose = TRUE) {
  mcol <- .metric_col(p)
  qc <- copy(qc_kept)
  qc[, phage_metric_value := get(mcol)]
  if (p$phage_metric == "count")
    warning("phage_metric='count' is not depth-corrected: large cells get called ",
            "infected for being large.")

  method <- p$phage_method
  if (identical(method, "auto")) {
    method <- if (any(qc$condition == "control")) "control_quantile" else "otsu"
    message("  phage_method='auto' -> using '", method, "'")
  }
  if (identical(method, "self_quantile"))
    warning("phage_method='self_quantile' is circular: it fixes the infected ",
            "fraction at ", round(100 * (1 - (p$self_q_infected %||% 0.9))),
            "% by construction.")

  cut_fun <- if (identical(method, "control_quantile"))
    function(d) .cut_control_quantile(d, p) else
    function(d) .cut_distribution(d, p, method)

  scope <- p$phage_cutoff_scope %||% "global"
  if (scope %in% c("per_sample", "per_timepoint")) {
    qc[, grp := if (scope == "per_sample") as.character(sample_id)
                else as.character(timepoint)]
    cuts <- rbindlist(lapply(split(qc, by = "grp"), function(d) {
      o <- cut_fun(d); o[, grp := d$grp[1]]; o }))
  } else {
    qc[, grp := "ALL"]
    cuts <- cut_fun(qc); cuts[, grp := "ALL"]
  }
  qc <- merge(qc, cuts, by = "grp", all.x = TRUE)

  if (!"valid_viral_probes" %in% names(qc))
    stop("valid_viral_probes column missing -- run stage_qc() first")
  if (anyNA(qc$valid_viral_probes))
    stop("valid_viral_probes contains NA; data.table would treat those as FALSE ",
         "and silently call the cells not-infected")

  qc[, infection := "uninfected"]
  qc[phage_metric_value > thr_ambiguous, infection := "ambiguous"]
  qc[phage_metric_value > thr_infected & valid_viral_probes &
       phage_umi >= p$min_phage_umi, infection := "infected"]
  qc[, infection := factor(infection, levels = c("uninfected", "ambiguous", "infected"))]

  bad <- qc[infection == "infected" &
              !(phage_metric_value > thr_infected & valid_viral_probes), .N]
  if (bad > 0) stop(bad, " cells labelled infected without meeting both criteria")

  if (verbose) {
    cat("\n================ THRESHOLD DECISION ================\n")
    for (g in cuts$grp) {
      cu <- cuts[grp == g]; vv <- qc[grp == g]
      sep <- assess_separability(vv$phage_metric_value, p$valley_adjust %||% 1)
      n_inf <- sum(vv$infection == "infected")
      cat("\n", g, "  (n = ", nrow(vv), " cells, metric = ", mcol, ")\n", sep = "")
      cat("  method used        : ", cu$method, "\n", sep = "")
      cat("  raw value          : ", signif(cu$q_infected_raw, 4), "\n", sep = "")
      cat("  floor              : ", p$min_percent_infected,
          if (isTRUE(cu$floor_binding_infected)) "   <- BINDING" else "   (not binding)",
          "\n", sep = "")
      cat("  thr_infected USED  : ", signif(cu$thr_infected, 4), "\n", sep = "")
      cat("  thr_ambiguous USED : ", signif(cu$thr_ambiguous, 4), "\n", sep = "")
      cat("  -> infected: ", n_inf, " / ", nrow(vv), " cells (",
          round(100 * n_inf / nrow(vv), 1), "%)\n", sep = "")
      cat("  separability       : ", sep$note, "\n", sep = "")
      if (!isTRUE(sep$bimodal))
        cat("  !! not bimodal here: the cutoff slices one population rather than\n",
            "     separating two. Read percent_MS2 as a continuum, not a call.\n", sep = "")
    }
    cat("\n===================================================\n")
  }

  qc <- add_infection_grade(qc, p, p$n_load_grades %||% 4, verbose)

  has_ctrl <- any(qc$condition == "control")
  fpr <- if (has_ctrl)
    qc[condition == "control",
       .(n_control_cells = .N,
         pct_called_infected  = round(100 * mean(infection == "infected"), 2),
         pct_called_ambiguous = round(100 * mean(infection == "ambiguous"), 2)),
       by = .(timepoint)][order(timepoint)] else NULL

  comp <- qc[, .N, by = .(sample_id, timepoint, condition, infection)]
  comp[, pct := round(100 * N / sum(N), 1), by = sample_id]

  crit <- qc[, .(n_cells = .N,
                 n_pass_metric = sum(phage_metric_value > thr_infected),
                 n_pass_probes = sum(valid_viral_probes),
                 n_pass_both   = sum(phage_metric_value > thr_infected & valid_viral_probes),
                 limiting = if (sum(valid_viral_probes) <
                                sum(phage_metric_value > thr_infected))
                              "distinct-probe rule" else "metric cutoff"),
             by = .(sample_id, timepoint, condition)][order(condition, timepoint)]

  list(qc = qc, cutoffs = cuts, control_fpr = fpr, criterion_limits = crit,
       composition = comp[order(timepoint, condition, infection)])
}

ambient_diagnostics <- function(qc_kept) {
  if (!any(qc_kept$condition == "control")) return(NULL)
  qc_kept <- qc_with_offset_umi(qc_kept)
  qc_kept[condition == "control", {
    ok <- host_umi > 0 & phage_umi > 0
    if (sum(ok) > 10) {
      fit <- lm(log1p(phage_umi[ok]) ~ log1p(host_umi[ok]))
      .(n_phage_pos = sum(ok),
        slope_logphage_vs_loghost = round(unname(coef(fit)[2]), 3),
        median_ratio = signif(median(phage_ratio), 3))
    } else .(n_phage_pos = sum(ok), slope_logphage_vs_loghost = NA_real_,
             median_ratio = NA_real_)
  }, by = .(sample_id, timepoint)][order(timepoint)]
}

In [ ]:
.get_counts <- function(obj, assay = "RNA") {
  if (utils::packageVersion("SeuratObject") >= "5.0.0")
    SeuratObject::LayerData(obj, assay = assay, layer = "counts")
  else Seurat::GetAssayData(obj, assay = assay, slot = "counts")
}
.set_data <- function(obj, m, assay = "RNA") {
  if (utils::packageVersion("SeuratObject") >= "5.0.0") {
    SeuratObject::LayerData(obj, assay = assay, layer = "data") <- m
  } else {
    obj <- Seurat::SetAssayData(obj, assay = assay, slot = "data", new.data = m)
  }
  obj
}

build_seurat <- function(counts, qc, p) {
  orig <- rownames(counts)
  new  <- make.unique(gsub("_", "-", orig))
  if (!identical(orig, new)) {
    if (grepl("_", p$phage_prefix))
      warning("phage_prefix '", p$phage_prefix, "' contains '_', which will NOT ",
              "match after renaming. Use '^MS2'.")
    stopifnot(sum(is_phage(new, p)) == sum(is_phage(orig, p)))
  }
  rownames(counts) <- new

  md <- as.data.frame(qc); rownames(md) <- md$cell
  md <- md[colnames(counts), , drop = FALSE]
  obj <- CreateSeuratObject(counts = counts, meta.data = md,
                            project = paste0("probacseq_", make_run_tag(p)),
                            min.cells = 0, min.features = 0)
  obj@misc$feature_map <- data.table(original = orig, seurat = new)
  obj
}

normalize_host <- function(obj, p) {
  cnt   <- .get_counts(obj)
  is_ph <- is_phage(rownames(cnt), p)
  denom <- if (identical(p$norm_basis, "host"))
             Matrix::colSums(cnt[!is_ph, , drop = FALSE]) else Matrix::colSums(cnt)
  if (any(denom <= 0)) {
    warning(sum(denom <= 0), " cells have zero host counts; size factor set to 1.")
    denom[denom <= 0] <- 1
  }
  sf <- p$scale_factor %||% as.numeric(median(denom))
  message("  normalization: basis=", p$norm_basis, "  scale.factor=", round(sf, 1))
  norm <- log1p(cnt %*% Matrix::Diagonal(x = sf / denom))
  dimnames(norm) <- dimnames(cnt)
  obj <- .set_data(obj, norm)
  obj$size_factor_denom <- denom
  obj@misc$scale_factor <- sf
  obj@misc$norm_basis   <- p$norm_basis
  obj
}

add_ms2_norm <- function(obj, p) {
  sf    <- obj@misc$scale_factor %||% 1
  denom <- obj$size_factor_denom
  if (is.null(denom)) stop("run normalize_host() before add_ms2_norm()")
  tot <- obj@meta.data[["phage_umi"]]
  if (is.null(tot)) stop("phage_umi missing from metadata")
  obj$MS2_norm <- as.numeric(log1p(tot * sf / denom))
  obj
}

select_hvg <- function(obj, p) {
  all_feats <- rownames(obj); is_ph <- is_phage(all_feats, p)
  obj <- FindVariableFeatures(obj, selection.method = "vst",
           nfeatures = min(length(all_feats), p$n_hvg + sum(is_ph)), verbose = FALSE)
  hvg <- VariableFeatures(obj)
  if (!p$hvg_include_phage) hvg <- setdiff(hvg, all_feats[is_ph])
  hvg <- head(hvg, p$n_hvg)
  VariableFeatures(obj) <- hvg
  message("  HVG: ", length(hvg), " features (phage ",
          if (p$hvg_include_phage) "included" else "excluded", ")")
  obj
}

run_dimred_cluster <- function(obj, p) {
  set.seed(p$seed)
  obj <- ScaleData(obj, features = VariableFeatures(obj),
                   vars.to.regress = p$vars_to_regress, verbose = FALSE)
  npcs <- min(p$npcs, ncol(obj) - 1, length(VariableFeatures(obj)) - 1)
  obj  <- RunPCA(obj, features = VariableFeatures(obj), npcs = npcs, verbose = FALSE)

  reduction <- "pca"
  if (identical(p$integrate, "harmony")) {
    if (!requireNamespace("harmony", quietly = TRUE))
      stop("integrate='harmony' requires the 'harmony' package")
    obj <- harmony::RunHarmony(obj, group.by.vars = p$harmony_group,
                               reduction.use = "pca", verbose = FALSE)
    reduction <- "harmony"
  }
  dims <- p$dims_use[p$dims_use <= npcs]
  k_nn <- max(2L, min(20L, ncol(obj) - 1L))
  u_nn <- max(2L, min(30L, ncol(obj) - 1L))
  obj <- FindNeighbors(obj, reduction = reduction, dims = dims,
                       k.param = k_nn, verbose = FALSE)
  obj <- FindClusters(obj, resolution = p$resolution, verbose = FALSE)
  obj <- RunUMAP(obj, reduction = reduction, dims = dims,
                 n.neighbors = u_nn, verbose = FALSE)
  obj@misc$reduction_used <- reduction; obj@misc$dims_used <- dims
  obj
}

find_markers <- function(obj, p) {
  feats <- rownames(obj)
  ph_i  <- is_phage(feats, p)
  Idents(obj) <- "seurat_clusters"
  if (nlevels(obj$seurat_clusters) < 2) {
    message("  only one cluster -- no marker DE to run")
    return(list(host = data.table(), phage = NULL))
  }
  mk <- FindAllMarkers(obj, features = feats[!ph_i], only.pos = p$marker_only_pos,
                       min.pct = p$marker_min_pct,
                       logfc.threshold = p$marker_logfc_thresh,
                       test.use = p$marker_test, verbose = FALSE)
  mk_phage <- if (sum(ph_i) >= 2) tryCatch(
      FindAllMarkers(obj, features = feats[ph_i], only.pos = FALSE, min.pct = 0,
                     logfc.threshold = 0, test.use = p$marker_test, verbose = FALSE),
      error = function(e) NULL) else NULL
  list(host = as.data.table(mk), phage = as.data.table(mk_phage))
}

cluster_composition <- function(obj) {
  md <- as.data.table(obj@meta.data)
  md[, cluster := as.character(seurat_clusters)]
  lev <- c("uninfected", "ambiguous", "infected")
  if (!is.factor(md$infection)) md[, infection := factor(infection, levels = lev)]
  lev <- levels(md$infection)

  comp <- dcast(md[, .N, by = .(cluster, infection)], cluster ~ infection,
                value.var = "N", fill = 0, drop = FALSE)
  for (l in setdiff(lev, names(comp))) comp[, (l) := 0L]
  setcolorder(comp, c("cluster", lev))
  comp[, n_cells := rowSums(.SD), .SDcols = lev]
  for (cn in lev) comp[, (paste0("pct_", cn)) := round(100 * get(cn) / n_cells, 1)]

  load_dt <- md[, .(median_percent_MS2 = round(median(percent_MS2), 2),
                    median_host_umi = as.numeric(median(host_umi))), by = cluster]
  comp <- merge(comp, load_dt, by = "cluster")
  comp[order(as.numeric(cluster))][]
}

In [ ]:
stage_read <- function(p) {
  ss  <- read_sample_sheet(p$sample_sheet)
  ss  <- resolve_load_set(ss, p)
  cat("reading ", nrow(ss), " sample(s): ",
      paste(ss$sample_id, collapse = ", "), "\n", sep = "")
  raw <- load_all_samples(ss, p)
  raw$sample_sheet <- ss
  raw
}

stage_aggregate <- function(raw, p, verbose = TRUE) {
  if (!identical(p$level, "gene")) {
    if (verbose) message("  level = 'probe' -- nothing to aggregate")
    return(NULL)
  }
  cg <- aggregate_probes(raw$counts_probe, p$agg_method, p)
  if (verbose)
    message("  ", nrow(raw$counts_probe), " probes -> ", nrow(cg),
            " genes using '", p$agg_method, "'  (probe matrix kept for MS2)")
  cg
}

.qc_basis_matrix <- function(basis, raw, counts_gene, what) {
  if (identical(basis, "gene")) {
    if (is.null(counts_gene)) {
      warning(what, " = 'gene' but level != 'gene'; falling back to probes.")
      return(raw$counts_probe)
    }
    return(counts_gene)
  }
  raw$counts_probe
}

stage_qc <- function(raw, p, counts_gene = NULL) {
  if (is.null(counts_gene)) counts_gene <- stage_aggregate(raw, p, verbose = FALSE)

  feat_basis  <- p$qc_feat_basis  %||% p$qc_basis %||% "probe"
  umi_basis   <- p$qc_umi_basis   %||% p$qc_basis %||% "probe"
  phage_basis <- p$qc_phage_basis %||% "probe"

  if (!identical(phage_basis, "probe"))
    warning("qc_phage_basis = '", phage_basis, "': percent_MS2, phage_per_host ",
            "and therefore every infection call will move. Use 'probe'.")

  message("  QC bases -- host_umi: ", umi_basis, " | host_feat: ", feat_basis,
          " | phage (and ratio denominator): ", phage_basis)

  qc <- compute_qc_metrics(
          .qc_basis_matrix(umi_basis, raw, counts_gene, "qc_umi_basis"),
          raw$meta, p,
          mat_feat  = .qc_basis_matrix(feat_basis,  raw, counts_gene, "qc_feat_basis"),
          mat_phage = .qc_basis_matrix(phage_basis, raw, counts_gene, "qc_phage_basis"))

  vvp <- compute_valid_viral_probes(raw$counts_probe, p)
  setkey(vvp, cell); setkey(qc, cell)
  qc <- vvp[qc]
  if (anyNA(qc$valid_viral_probes))
    stop(sum(is.na(qc$valid_viral_probes)), " cells have NA valid_viral_probes")

  thr  <- compute_qc_thresholds(qc, p)
  filt <- apply_qc_filter(qc, thr, p)
  list(qc = filt$qc, thresholds = thr, summary = filt$summary,
       counts_gene = counts_gene,
       bases = list(umi = umi_basis, feat = feat_basis, phage = phage_basis))
}

make_working_matrix <- function(counts_probe, cells, p, counts_gene = NULL) {
  if (identical(p$level, "gene")) {
    if (is.null(counts_gene)) {
      counts_gene <- aggregate_probes(counts_probe[, cells, drop = FALSE],
                                      p$agg_method, p)
      cw <- counts_gene
    } else {
      miss <- setdiff(cells, colnames(counts_gene))
      if (length(miss))
        stop(length(miss), " kept cells are missing from the gene matrix")
      cw <- counts_gene[, cells, drop = FALSE]
    }
    message("  ", nrow(cw), " genes x ", format(ncol(cw), big.mark = ","),
            " cells (agg = '", p$agg_method, "')")
  } else {
    cw <- counts_probe[, cells, drop = FALSE]
    message("  keeping ", nrow(cw), " probes")
  }
  drop_outlier_features(cw, p)
}

analyze_one <- function(counts_work, qc_sub, p, outdir = NULL, label = "sample") {
  tabdir <- NULL
  if (!is.null(outdir)) {
    tabdir <- file.path(outdir, "tables")
    dir.create(tabdir, recursive = TRUE, showWarnings = FALSE)
  }

  cw  <- filter_features(counts_work[, qc_sub$cell, drop = FALSE], p)
  obj <- build_seurat(cw, qc_sub, p)
  obj <- normalize_host(obj, p)
  obj <- add_ms2_norm(obj, p)
  obj <- select_hvg(obj, p)
  obj <- run_dimred_cluster(obj, p)
  message("  clusters found: ", nlevels(obj$seurat_clusters))
  obj@misc$analysis_label <- label

  mk <- tryCatch(find_markers(obj, p), error = function(e) {
    message("  !! find_markers failed: ", conditionMessage(e))
    list(host = data.table(), phage = NULL) })
  comp <- tryCatch(cluster_composition(obj), error = function(e) {
    message("  !! cluster_composition failed: ", conditionMessage(e)); NULL })

  top10 <- if (nrow(mk$host))
    mk$host[order(cluster, -avg_log2FC), head(.SD, 10), by = cluster] else data.table()

  if (!is.null(outdir)) {
    saveRDS(obj, file.path(outdir, paste0("seurat_", make_run_tag(p), "_", label, ".rds")))
    if (nrow(mk$host)) fwrite(mk$host, file.path(tabdir, "markers_host_per_cluster.csv"))
    if (nrow(top10))   fwrite(top10,   file.path(tabdir, "markers_host_top10.csv"))
    if (!is.null(mk$phage) && nrow(mk$phage))
      fwrite(mk$phage, file.path(tabdir, "markers_phage_per_cluster.csv"))
    if (!is.null(comp))   fwrite(comp,   file.path(tabdir, "cluster_composition.csv"))
  }
  list(obj = obj, markers = mk, top10 = top10, composition = comp, label = label)
}

.print_analysis <- function(a) {
  if (nrow(a$top10)) {
    cat("\ntop cluster markers:\n")
    print(a$top10[, .(cluster, gene, avg_log2FC = round(avg_log2FC, 2),
                      pct.1, pct.2, p_val_adj)])
  }
  if (!is.null(a$composition)) {
    cat("\ncluster composition:\n"); print(a$composition)
  }
}

stage_analyze_per_sample <- function(counts_work, qc_final, p, root = NULL) {
  ids <- p$analysis_samples %||% unique(qc_final$sample_id)
  if (identical(ids, "all")) ids <- unique(qc_final$sample_id)
  ids <- intersect(ids, unique(qc_final$sample_id))
  out <- vector("list", length(ids)); names(out) <- ids

  for (sid in ids) {
    qc_sub <- qc_final[sample_id == sid]
    message("\n---------------------------------------------")
    message("  sample ", sid, "  (", nrow(qc_sub), " cells)")
    message("---------------------------------------------")
    if (nrow(qc_sub) < p$min_cells_per_analysis) {
      message("  !! skipped: fewer than ", p$min_cells_per_analysis, " cells")
      next
    }
    sdir <- if (is.null(root)) NULL else file.path(root, "per_sample", sid)
    if (!is.null(sdir)) dir.create(sdir, recursive = TRUE, showWarnings = FALSE)
    out[[sid]] <- tryCatch({
      a <- analyze_one(counts_work, qc_sub, p, sdir, label = sid)
      .print_analysis(a); a
    }, error = function(e) {
      message("  !! sample ", sid, " failed: ", conditionMessage(e),
              "\n     (other samples continue)")
      NULL })
    gc(verbose = FALSE)
  }
  out[!vapply(out, is.null, logical(1))]
}

summarize_per_sample <- function(per_sample) {
  rbindlist(lapply(per_sample, function(a) {
    md <- as.data.table(a$obj@meta.data)
    getn <- function(k) sum(md$infection == k)
    data.table(
      sample_id = a$label, timepoint = md$timepoint[1],
      condition = as.character(md$condition[1]),
      n_cells = ncol(a$obj), n_features = nrow(a$obj),
      n_hvg = length(VariableFeatures(a$obj)),
      n_clusters = nlevels(a$obj$seurat_clusters),
      median_host_umi = as.numeric(median(md$host_umi)),
      median_percent_MS2 = round(median(md$percent_MS2), 2),
      n_uninfected = getn("uninfected"), n_ambiguous = getn("ambiguous"),
      n_infected = getn("infected"),
      pct_infected = round(100 * getn("infected") / nrow(md), 2),
      n_cluster_markers = if (!is.null(a$markers$host)) nrow(a$markers$host) else 0L)
  }), fill = TRUE)[order(condition, timepoint)]
}

run_pipeline <- function(p, cached_raw = NULL) {
  set.seed(p$seed)
  run_tag <- make_run_tag(p)
  p$outlier_genes <- resolve_outliers(p)
  if (length(p$outlier_genes))
    message("outlier genes listed: ", length(p$outlier_genes))
  outdir <- tabdir <- NULL
  if (isTRUE(p$save_outputs)) {
    outdir <- file.path(p$out_root, run_tag)
    tabdir <- file.path(outdir, "tables")
    dir.create(tabdir, recursive = TRUE, showWarnings = FALSE)
  }

  message("\n============================================================")
  message("ProBac-seq  |  level = ", p$level,
          if (p$level == "gene") paste0(" (", p$agg_method, ")") else "",
          "  |  out = ", outdir %||% "<not saved>")
  message("============================================================")

  message("\n[1] reading data")
  raw <- cached_raw %||% stage_read(p)
  diagnose_feature_names(rownames(raw$counts_probe), p)

  message("\n[2] probe -> gene aggregation")
  counts_gene <- stage_aggregate(raw, p)

  message("\n[3] QC and cell filtering")
  qcres <- stage_qc(raw, p, counts_gene = counts_gene)
  print(qcres$thresholds); print(qcres$summary)
  qc_kept <- qcres$qc[keep == TRUE]
  message("  kept ", nrow(qc_kept), " / ", nrow(qcres$qc), " barcodes")
  if (nrow(qc_kept) < 50) stop("Too few cells survived QC -- check the thresholds.")

  message("\n[4] infection classification (metric = ", p$phage_metric,
          ", method = ", p$phage_method, ")")
  cls <- classify_infection(qc_kept, p)
  cls$ambient <- ambient_diagnostics(qc_kept)
  cat("\ncutoffs:\n");                            print(cls$cutoffs)
  cat("\nwhich criterion limits the call:\n");    print(cls$criterion_limits)
  if (!is.null(cls$control_fpr)) {
    cat("\nempirical FALSE-POSITIVE RATE in controls:\n"); print(cls$control_fpr)
  } else {
    cat("\nNO CONTROL SAMPLES: no empirical false-positive rate. Specificity\n",
        "rests on the ", p$min_valid_probes, "-distinct-probe rule.\n", sep = "")
  }
  cat("\ncalls per sample:\n"); print(cls$composition)

  message("\n[5] building working matrix at level = ", p$level)
  counts_work <- make_working_matrix(raw$counts_probe, cls$qc$cell, p,
                                     counts_gene = qcres$counts_gene)

  p_run <- p
  if (identical(p$scale_factor_scope, "global") && is.null(p$scale_factor)) {
    hb <- is_phage(rownames(counts_work), p)
    p_run$scale_factor <- as.numeric(median(
      Matrix::colSums(counts_work[!hb, , drop = FALSE])))
    message("  shared scale.factor = ", round(p_run$scale_factor, 1))
  }

  message("\n[5b] per-sample analyses (independent normalization, PCA, clustering)")
  per_sample <- stage_analyze_per_sample(counts_work, cls$qc, p_run, outdir)
  smry <- NULL
  if (length(per_sample)) {
    smry <- summarize_per_sample(per_sample)
    cat("\n=== per-sample summary ===\n"); print(smry)
  } else warning("No per-sample analysis succeeded.")

  if (isTRUE(p$save_outputs)) {
    fwrite(qcres$thresholds, file.path(tabdir, "qc_thresholds.csv"))
    fwrite(qcres$summary,    file.path(tabdir, "qc_summary_per_sample.csv"))
    fwrite(cls$cutoffs,      file.path(tabdir, "phage_cutoffs.csv"))
    fwrite(cls$composition,  file.path(tabdir, "infection_calls_per_sample.csv"))
    fwrite(cls$criterion_limits, file.path(tabdir, "phage_criterion_limits.csv"))
    if (!is.null(cls$control_fpr))
      fwrite(cls$control_fpr, file.path(tabdir, "phage_control_fpr.csv"))
    fwrite(cls$qc[, .(cell, sample_id, timepoint, condition,
                      host_umi, host_feat, host_umi_probe,
                      phage_umi, phage_feat, n_valid_probes, valid_viral_probes,
                      percent_MS2, phage_per_host, phage_metric_value,
                      thr_ambiguous, thr_infected, infection, phage_load_grade)],
           file.path(tabdir, "infection_calls_per_cell.csv.gz"))
    if (!is.null(smry)) fwrite(smry, file.path(outdir, "per_sample_summary.csv"))
    saveRDS(p_run, file.path(outdir, "params_used.rds"))
    message("\nWritten to: ", normalizePath(outdir))
  }

  invisible(list(params = p_run, run_tag = run_tag, raw = raw,
                 qc = qcres$qc, thresholds = qcres$thresholds,
                 qc_summary = qcres$summary, classification = cls,
                 counts_work = counts_work, per_sample = per_sample,
                 per_sample_summary = smry, outdir = outdir))
}

In [ ]:
CLUSTER_PALETTE <- c("#33A02C", "#6A3D9B", "#E7298A", "#595959",
                     "#8FBC45", "#C2255C", "#7F7F7F", "#4F7942",
                     "#D782C0", "#9B2242", "#5D3A8E", "#6B8E23")

paper_theme <- function(base_size = 14,
                        axis_style = c("labels", "classic", "none", "arrow")) {
  axis_style <- match.arg(axis_style)
  th <- ggplot2::theme_bw(base_size = base_size) +
    ggplot2::theme(
      panel.grid        = ggplot2::element_blank(),
      panel.background  = ggplot2::element_rect(fill = "white", colour = NA),
      plot.background   = ggplot2::element_rect(fill = "white", colour = NA),
      plot.title        = ggplot2::element_text(face = "plain", hjust = 0.5,
                            size = base_size * 1.2,
                            margin = ggplot2::margin(b = 5)),
      plot.subtitle     = ggplot2::element_text(hjust = 0.5, colour = "grey30",
                            size = base_size * 0.95,
                            margin = ggplot2::margin(b = 4)),
      legend.title      = ggplot2::element_text(size = base_size * 1.0,
                            face = "plain", lineheight = 1.05),
      legend.text       = ggplot2::element_text(size = base_size * 0.95),
      legend.background = ggplot2::element_blank(),
      plot.caption      = ggplot2::element_text(hjust = 1, colour = "grey25",
                            size = base_size * 0.75,
                            margin = ggplot2::margin(t = 2)),
      plot.margin       = ggplot2::margin(6, 6, 6, 6))

  axis_on <- ggplot2::theme(
    axis.title = ggplot2::element_text(size = base_size * 1.05, face = "plain"),
    axis.text  = ggplot2::element_text(size = base_size * 0.9, colour = "grey20"),
    axis.ticks = ggplot2::element_line(colour = "grey20", linewidth = 0.4))
  axis_off <- ggplot2::theme(
    axis.title = ggplot2::element_blank(),
    axis.text  = ggplot2::element_blank(),
    axis.ticks = ggplot2::element_blank())

  th + switch(axis_style,
    "classic" = axis_on + ggplot2::theme(
        panel.border = ggplot2::element_blank(),
        axis.line    = ggplot2::element_line(colour = "black", linewidth = 0.5)),
    "labels"  = axis_on + ggplot2::theme(
        panel.border = ggplot2::element_rect(colour = "grey20", linewidth = 0.6)),
    "arrow"   = axis_off + ggplot2::theme(panel.border = ggplot2::element_blank()),
    "none"    = axis_off + ggplot2::theme(panel.border = ggplot2::element_blank()))
}

.colourbar <- function(title, base_size)
  ggplot2::guide_colourbar(
    title.position = "top", frame.colour = "grey25", frame.linewidth = 0.4,
    ticks.colour = "grey25",
    barwidth  = grid::unit(base_size * 0.75, "pt"),
    barheight = grid::unit(base_size * 7.5, "pt"))

.auto_point_size <- function(n)
  if (n < 2000) 1.5 else if (n < 8000) 0.9 else
  if (n < 20000) 0.6 else if (n < 60000) 0.4 else 0.28

.pts <- function(mapping, size, alpha, rasterise, dpi) {
  g <- ggplot2::geom_point(mapping, size = size, alpha = alpha, stroke = 0)
  if (isTRUE(rasterise) && requireNamespace("ggrastr", quietly = TRUE))
    g <- ggrastr::rasterise(g, dpi = dpi)
  g
}

umap_table <- function(res, sid, reduction = "umap", extra = NULL) {
  a <- res$per_sample[[sid]]
  if (is.null(a))
    stop("'", sid, "' is not in res$per_sample. Available: ",
         paste(names(res$per_sample), collapse = ", "))
  o <- a$obj
  if (!reduction %in% names(o@reductions))
    stop("reduction '", reduction, "' missing from ", sid)
  em <- Seurat::Embeddings(o, reduction)[, 1:2, drop = FALSE]
  d <- data.table::data.table(
    cell    = rownames(em),
    x       = em[, 1], y = em[, 2],
    cluster = factor(o$seurat_clusters, levels = levels(o$seurat_clusters)),
    percent_MS2 = as.numeric(o$percent_MS2),
    phage_per_host = as.numeric(o$phage_per_host),
    infection = o$infection)
  if (!is.null(extra))
    for (nm in names(extra))
      data.table::set(d, j = nm, value = as.numeric(extra[[nm]][d$cell]))
  d[is.finite(x) & is.finite(y)][]
}

.axis_limits <- function(d, pad = 0.04, square = TRUE) {
  rx <- range(d$x); ry <- range(d$y)
  if (square) {
    span <- max(diff(rx), diff(ry))
    if (span == 0) span <- 1
    cx <- mean(rx); cy <- mean(ry)
    rx <- cx + c(-span, span) / 2
    ry <- cy + c(-span, span) / 2
  }
  px <- diff(rx) * pad; py <- diff(ry) * pad
  if (px == 0) px <- 0.04
  if (py == 0) py <- 0.04
  list(x = rx + c(-px, px), y = ry + c(-py, py))
}

panel_umap <- function(d, mode = c("cluster", "value"), value_col = "value",
                       title = NULL, subtitle = NULL, caption = NULL,
                       limits = NULL, palette = "viridis", direction = 1,
                       legend_title = NULL, breaks = ggplot2::waiver(),
                       labels = ggplot2::waiver(), trans = NULL,
                       cluster_labels = TRUE, cluster_palette = NULL,
                       point_size = NULL, point_alpha = 0.9,
                       base_size = 14, axis_style = "labels",
                       rasterise = TRUE, dpi = 400, lims = NULL,
                       show_axis_titles = TRUE) {
  mode <- match.arg(mode)
  if (is.null(lims)) lims <- .axis_limits(d)
  if (is.null(point_size)) point_size <- .auto_point_size(nrow(d))

  base <- list(
    ggplot2::coord_fixed(xlim = lims$x, ylim = lims$y, expand = FALSE, clip = "off"),
    ggplot2::labs(x = if (show_axis_titles) "UMAP 1" else NULL,
                  y = if (show_axis_titles) "UMAP 2" else NULL,
                  title = title, subtitle = subtitle, caption = caption),
    paper_theme(base_size, axis_style))

  if (mode == "cluster") {
    nk <- nlevels(d$cluster)
    cols <- cluster_palette %||%
      (if (nk <= length(CLUSTER_PALETTE)) CLUSTER_PALETTE[seq_len(nk)]
       else grDevices::colorRampPalette(CLUSTER_PALETTE)(nk))
    p <- ggplot2::ggplot(d) +
      .pts(ggplot2::aes(x, y, colour = cluster), point_size, point_alpha,
           rasterise, dpi) +
      ggplot2::scale_colour_manual(values = cols,
                                   name = legend_title %||% "Cluster") + base
    if (cluster_labels) {
      cen <- d[, .(x = stats::median(x), y = stats::median(y)), by = cluster]
      p <- p + ggplot2::geom_text(data = cen, ggplot2::aes(x, y, label = cluster),
            fontface = "plain", size = base_size * 0.36, colour = "grey10") +
        ggplot2::theme(legend.position = "none")
    } else {
      p <- p + ggplot2::guides(colour = ggplot2::guide_legend(
        override.aes = list(size = 3, alpha = 1), ncol = 1))
    }
  } else {
    dd <- data.table::copy(d)
    data.table::set(dd, j = ".v", value = as.numeric(dd[[value_col]]))
    if (!is.null(trans) && !is.null(limits))
      data.table::set(dd, j = ".v",
        value = pmin(pmax(dd$.v, min(limits)), max(limits)))
    data.table::setorderv(dd, ".v")        # high values drawn last, on top

    sc <- list(limits = limits, oob = scales::squish,
               breaks = breaks, labels = labels,
               na.value = "grey85", name = legend_title %||% "",
               guide = .colourbar(legend_title %||% "", base_size))
    sc[[if (utils::packageVersion("ggplot2") >= "3.5.0") "transform" else "trans"]] <-
      trans %||% "identity"

    custom <- length(palette) > 1L || grepl("^#", palette[1])
    if (custom) {
      sc$colours <- if (direction < 0) rev(palette) else palette
      scale_fn   <- ggplot2::scale_colour_gradientn
    } else {
      sc$option <- palette; sc$direction <- direction
      scale_fn  <- ggplot2::scale_colour_viridis_c
    }

    p <- ggplot2::ggplot(dd) +
      .pts(ggplot2::aes(x, y, colour = .v), point_size, point_alpha,
           rasterise, dpi) +
      do.call(scale_fn, sc) + base
  }

  if (identical(axis_style, "arrow")) {
    rx <- lims$x; ry <- lims$y
    ox <- rx[1] + diff(rx) * 0.02; oy <- ry[1] + diff(ry) * 0.02
    ln <- diff(rx) * 0.18
    p <- p +
      ggplot2::annotate("segment", x = ox, xend = ox + ln, y = oy, yend = oy,
        arrow = grid::arrow(length = grid::unit(5, "pt"), type = "closed"),
        linewidth = 0.45, colour = "grey25") +
      ggplot2::annotate("segment", x = ox, xend = ox, y = oy,
        yend = oy + diff(ry) * 0.18,
        arrow = grid::arrow(length = grid::unit(5, "pt"), type = "closed"),
        linewidth = 0.45, colour = "grey25") +
      ggplot2::annotate("text", x = ox + ln * 0.55, y = oy - diff(ry) * 0.035,
        label = "UMAP 1", size = base_size * 0.24, colour = "grey25", vjust = 1) +
      ggplot2::annotate("text", x = ox - diff(rx) * 0.035,
        y = oy + diff(ry) * 0.09, label = "UMAP 2", angle = 90,
        size = base_size * 0.24, colour = "grey25", hjust = 0.5)
  }
  p
}

row_label_panel <- function(txt, base_size = 14, colour = "grey10") {
  ggplot2::ggplot() +
    ggplot2::annotate("text", x = 0, y = 0, label = txt, angle = 90,
                      fontface = "plain", size = base_size * 0.38, colour = colour) +
    ggplot2::theme_void()
}

pretty_sample <- function(sid) {
  n <- regmatches(sid, regexpr("[0-9]+", sid))
  if (!length(n) || !nzchar(n)) return(sid)
  is_ctrl <- grepl("ctrl|control|mock", sid, ignore.case = TRUE)
  paste0(if (is_ctrl) "Control, " else "MS2-infected, ", n, " min")
}

save_paper_fig <- function(fig, name, width, height, dpi = 400,
                           formats = FIG_FORMATS, dir = FIG_DIR, show = TRUE) {
  dir.create(dir, recursive = TRUE, showWarnings = FALSE)
  for (fmt in formats) {
    f <- file.path(dir, paste0(name, ".", fmt))
    ok <- tryCatch({
      if (identical(fmt, "pdf") && isTRUE(capabilities("cairo")))
        ggplot2::ggsave(f, fig, width = width, height = height, dpi = dpi,
                        bg = "white", device = grDevices::cairo_pdf)
      else
        ggplot2::ggsave(f, fig, width = width, height = height, dpi = dpi,
                        bg = "white")
      TRUE
    }, error = function(e) { message("  !! could not write ", f, ": ",
                                     conditionMessage(e)); FALSE })
    if (ok) message("wrote ", f, "  (", width, " x ", height, " in)")
  }
  if (isTRUE(show)) {
    options(repr.plot.width = width, repr.plot.height = height)
    print(fig)
  }
  invisible(fig)
}

In [ ]:
shared_ms2_limits <- function(specs, feature = "percent_MS2",
                              clip = c(0.01, 0.99), round_to = 5) {
  v <- unlist(lapply(specs, function(s) {
    d <- umap_table(s$res, s$sample)
    as.numeric(d[[feature]])
  }))
  v <- v[is.finite(v)]
  lim <- as.numeric(stats::quantile(v, clip, na.rm = TRUE))
  if (!is.null(round_to) && round_to > 0)
    lim <- c(floor(lim[1] / round_to) * round_to, ceiling(lim[2] / round_to) * round_to)
  lim <- c(max(0, lim[1]), min(100, lim[2]))
  cat("shared MS2 colour scale: ", lim[1], "% to ", lim[2], "%",
      "   (", clip[1] * 100, "-", clip[2] * 100, "th percentile of ",
      format(length(v), big.mark = ","), " pooled cells; observed ",
      round(min(v), 1), "-", round(max(v), 1), "%)\n", sep = "")
  for (s in specs) {
    d <- umap_table(s$res, s$sample)
    cat("   ", s$sample, ": n = ", nrow(d), ", median ",
        round(median(d[[feature]]), 1), "%, IQR ",
        round(as.numeric(quantile(d[[feature]], .25)), 1), "-",
        round(as.numeric(quantile(d[[feature]], .75)), 1), "%\n", sep = "")
  }
  lim
}

.ms2_scale_bits <- function(limits, breaks = NULL) {
  brk <- breaks %||% pretty(limits, n = 5)
  brk <- brk[brk >= limits[1] & brk <= limits[2]]
  list(breaks = brk, labels = function(z) paste0(signif(z, 2), "%"))
}

logit_percent_trans <- function() {
  nt <- if ("new_transform" %in% getNamespaceExports("scales"))
          scales::new_transform else scales::trans_new
  nt("logit_percent",
     transform = function(x) stats::qlogis(x / 100),
     inverse   = function(x) 100 * stats::plogis(x),
     domain    = c(0, 100))
}

.add_fig_title <- function(fig, fig_title, base_size = 14) {
  if (is.null(fig_title) || !nzchar(fig_title)) return(fig)
  if (!requireNamespace("patchwork", quietly = TRUE)) {
    warning("'patchwork' is not installed -- fig_title dropped")
    return(fig)
  }
  if (!inherits(fig, "patchwork")) fig <- patchwork::wrap_plots(fig)
  fig + patchwork::plot_annotation(
    title = fig_title,
    theme = ggplot2::theme(
      plot.title = ggplot2::element_text(face = "plain", hjust = 0.5,
                     size = base_size * 1.3,
                     margin = ggplot2::margin(b = 8)),
      plot.background = ggplot2::element_rect(fill = "white", colour = NA)))
}

fig_clusters_ms2 <- function(res, sample, ms2_limits,
                             ms2_trans = NULL, ms2_breaks = NULL,
                             feature = "percent_MS2",
                             palette = "viridis", base_size = 14,
                             axis_style = "labels", panel_titles = NULL,
                             fig_title = NULL,
                             legend_title = "MS2\n(% of UMIs)",
                             show_median = FALSE, tags = FALSE,
                             point_size = NULL, point_alpha = 0.9,
                             cluster_labels = TRUE, rasterise = TRUE, dpi = 400,
                             width = NULL, height = NULL, name = NULL,
                             show = TRUE) {
  if (!requireNamespace("patchwork", quietly = TRUE))
    stop("'patchwork' is required to combine the panels")
  d <- umap_table(res, sample)
  d[, value := as.numeric(get(feature))]
  lims <- .axis_limits(d)                       # both panels, identical axes
  ttl  <- panel_titles %||% c("Clusters", "MS2 fraction")
  sb   <- .ms2_scale_bits(ms2_limits, ms2_breaks)

  pA <- panel_umap(d, "cluster", title = ttl[1], lims = lims,
                   cluster_labels = cluster_labels, base_size = base_size,
                   axis_style = axis_style, point_size = point_size,
                   point_alpha = point_alpha, rasterise = rasterise, dpi = dpi)
  pB <- panel_umap(d, "value", value_col = "value", title = ttl[2], lims = lims,
                   limits = ms2_limits, palette = palette,
                   legend_title = legend_title, breaks = sb$breaks,
                   labels = sb$labels, trans = ms2_trans, base_size = base_size,
                   axis_style = axis_style, point_size = point_size,
                   point_alpha = point_alpha, rasterise = rasterise, dpi = dpi,
                   caption = if (show_median)
                     paste0("median ", round(median(d$value), 1), "%") else NULL)
  if (tags) {
    pA <- pA + ggplot2::labs(tag = "A")
    pB <- pB + ggplot2::labs(tag = "B")
  }
  fig <- patchwork::wrap_plots(pA, pB, nrow = 1) &
    ggplot2::theme(plot.tag = ggplot2::element_text(face = "plain",
                     size = base_size * 1.15))
  fig <- .add_fig_title(fig, fig_title, base_size)

  width  <- width  %||% 8.6
  height <- height %||% (if (is.null(fig_title)) 4.3 else 4.7)
  name   <- name   %||% paste0("Fig2A_clusters_MS2_", sample)
  save_paper_fig(fig, name, width, height, dpi, show = show)
}

fig_ms2_only <- function(res, sample, ms2_limits,
                         ms2_trans = NULL, ms2_breaks = NULL,
                         feature = "percent_MS2",
                         palette = "viridis", base_size = 14,
                         axis_style = "labels", panel_title = "MS2 fraction",
                         fig_title = NULL,
                         legend_title = "MS2\n(% of UMIs)",
                         show_median = FALSE, point_size = NULL,
                         point_alpha = 0.9, rasterise = TRUE, dpi = 400,
                         width = NULL, height = NULL, name = NULL, show = TRUE) {
  d <- umap_table(res, sample)
  d[, value := as.numeric(get(feature))]
  sb <- .ms2_scale_bits(ms2_limits, ms2_breaks)

  p <- panel_umap(d, "value", value_col = "value", title = panel_title,
                  lims = .axis_limits(d), limits = ms2_limits, palette = palette,
                  legend_title = legend_title, breaks = sb$breaks,
                  labels = sb$labels, trans = ms2_trans, base_size = base_size,
                  axis_style = axis_style, point_size = point_size,
                  point_alpha = point_alpha, rasterise = rasterise, dpi = dpi,
                  caption = if (show_median)
                    paste0("median ", round(median(d$value), 1), "%") else NULL)

  p <- .add_fig_title(p, fig_title, base_size)

  width  <- width  %||% 5.0
  height <- height %||% (if (is.null(fig_title)) 4.3 else 4.7)
  name   <- name   %||% paste0("Fig2D_MS2_", sample)
  save_paper_fig(p, name, width, height, dpi, show = show)
}

In [ ]:
nb_glm <- function(y, grp, host) {
  if (!requireNamespace("MASS", quietly = TRUE)) stop("needs the 'MASS' package")
  d <- data.frame(y = as.numeric(y), g = as.numeric(grp), off = log(pmax(host, 1)))
  if (sum(d$y) == 0 || length(unique(d$g)) < 2)
    return(list(log2FC = NA_real_, ci_lo = NA_real_, ci_hi = NA_real_,
                p = NA_real_, model = "skipped"))
  fit <- try(suppressWarnings(MASS::glm.nb(y ~ g + offset(off), data = d)),
             silent = TRUE)
  if (!inherits(fit, "try-error")) {
    s  <- summary(fit)$coefficients
    ci <- try(suppressWarnings(stats::confint.default(fit)), silent = TRUE)
    return(list(log2FC = unname(coef(fit)[["g"]]) / log(2),
      ci_lo = if (!inherits(ci, "try-error")) ci["g", 1] / log(2) else NA_real_,
      ci_hi = if (!inherits(ci, "try-error")) ci["g", 2] / log(2) else NA_real_,
      p = unname(s["g", "Pr(>|z|)"]), model = "nb"))
  }
  fit2 <- try(suppressWarnings(stats::glm(y ~ g + offset(off),
                family = stats::quasipoisson, data = d)), silent = TRUE)
  if (inherits(fit2, "try-error"))
    return(list(log2FC = NA_real_, ci_lo = NA_real_, ci_hi = NA_real_,
                p = NA_real_, model = "failed"))
  s <- summary(fit2)$coefficients
  list(log2FC = unname(coef(fit2)[["g"]]) / log(2), ci_lo = NA_real_,
       ci_hi = NA_real_, p = unname(s["g", "Pr(>|t|)"]), model = "quasipoisson")
}

REGULON_PATTERNS <- c(glp = "^glp", mal = "^mal")

REGULON_EXTRA   <- list()                # e.g. list(mal = "lamB")
REGULON_EXCLUDE <- character(0)          # e.g. "glpR"

REGULON_LABELS <- c(glp = "glp regulon", mal = "mal regulon")
REGULON_SUB    <- c(glp = "glycerol / G3P", mal = "maltose / maltodextrin")

panel_genes <- function(res, pattern, p = res$params,
                        extra = character(0), exclude = REGULON_EXCLUDE) {
  feats <- rownames(res$raw$counts_probe)
  gn    <- unique(probe_to_gene(feats[!is_phage(feats, p)], p))
  hit   <- c(gn[grepl(pattern, gn, ignore.case = TRUE)],
             intersect(gn, extra))
  drop  <- c(p$outlier_genes %||% character(0), exclude)
  sort(unique(hit[!tolower(hit) %in% tolower(drop)]))
}

BYSTANDER_CLASSES <- c("uninfected", "ambiguous")
GROUP_LABEL       <- "uninfected exposed"

COL_CONTROL   <- "#7f8c9b"
COL_BYSTANDER <- "#e07b1a"
REGULON_FILL  <- c(glp = "#cfcfcf", mal = "#636363")   # greyscale

.sig_stars <- function(pv) vapply(pv, function(x)
  if (is.na(x)) "ns" else if (x < 1e-3) "***" else if (x < 1e-2) "**" else
  if (x < 0.05) "*" else "ns", character(1))

.tp_label <- function(tp) paste0(tp, " minutes")

.probe_rows <- function(res, genes, p = res$params) {
  og <- p$outlier_genes %||% character(0)
  genes <- setdiff(genes, genes[tolower(genes) %in% tolower(og)])
  gn <- probe_to_gene(rownames(res$raw$counts_probe), p)
  which(tolower(gn) %in% tolower(genes) &
          !is_phage(rownames(res$raw$counts_probe), p))
}

regulon_present <- function(res, genes, p = res$params, label = "", verbose = TRUE) {
  og <- p$outlier_genes %||% character(0)
  dropped <- genes[tolower(genes) %in% tolower(og)]
  genes   <- setdiff(genes, dropped)
  gn      <- probe_to_gene(rownames(res$raw$counts_probe), p)
  present <- genes[tolower(genes) %in% tolower(gn)]
  missing <- setdiff(genes, present)
  if (verbose) {
    cat("  ", label, ": ", length(present), "/", length(genes) + length(dropped),
        " gene(s) on the panel, ", length(.probe_rows(res, present, p)),
        " probe(s)\n", sep = "")
    cat("        present: ", paste(sort(present), collapse = ", "), "\n", sep = "")
    if (length(missing))
      cat("        NOT on the panel: ", paste(missing, collapse = ", "), "\n", sep = "")
    if (length(dropped))
      cat("        dropped as outlier gene(s): ",
          paste(dropped, collapse = ", "), "\n", sep = "")
  }
  present
}

.timepoint_pair <- function(res, tp, p = res$params) {
  ss   <- read_sample_sheet(p$sample_sheet)
  have <- unique(as.character(res$classification$qc$sample_id))
  inf  <- intersect(ss[timepoint == tp & condition == "infected", sample_id], have)
  ctl  <- intersect(ss[timepoint == tp & condition == "control",  sample_id], have)
  if (length(inf) != 1 || length(ctl) != 1)
    stop("timepoint ", tp, " needs exactly one infected and one control sample ",
         "among the samples that were read.\n  infected: ",
         paste(inf, collapse = ", ") %||% "<none>", "\n  control:  ",
         paste(ctl, collapse = ", ") %||% "<none>",
         "\n  read:     ", paste(have, collapse = ", "),
         "\n  Run ensure_samples_read() first, or set RUN_SAMPLES <- \"all\".")
  list(infected = inf, control = ctl)
}

.assert_calls_unchanged <- function(qc_old, qc_new, samples) {
  a <- qc_old[sample_id %in% samples, .(cell, old = as.character(infection))]
  b <- qc_new[sample_id %in% samples, .(cell, new = as.character(infection))]
  m <- merge(a, b, by = "cell", all = TRUE)
  n_lost <- sum(is.na(m$new)); n_gain <- sum(is.na(m$old))
  n_diff <- m[!is.na(old) & !is.na(new) & old != new, .N]
  if (n_lost || n_gain || n_diff)
    stop("re-reading changed the infection calls of samples that were already ",
         "loaded (", n_diff, " reclassified, ", n_lost, " lost, ", n_gain,
         " gained). The cutoff should be identical -- do not use these results.")
  message("  calls for the ", length(samples), " sample(s) already loaded are ",
          "unchanged (", nrow(m), " cells checked)")
  invisible(TRUE)
}

ensure_samples_read <- function(res, samples, p = res$params, verbose = TRUE) {
  have <- unique(as.character(res$classification$qc$sample_id))
  need <- setdiff(samples, have)
  if (!length(need)) {
    message("all requested samples are already loaded: ",
            paste(samples, collapse = ", "))
    return(res)
  }

  ss_all <- read_sample_sheet(p$sample_sheet)
  unknown <- setdiff(need, ss_all$sample_id)
  if (length(unknown))
    stop("not in the sample sheet: ", paste(unknown, collapse = ", "))
  ss_add <- ss_all[sample_id %in% need]
  gone   <- ss_add[!file.exists(path), sample_id]
  if (length(gone))
    stop("path does not exist for: ", paste(gone, collapse = ", "))

  message("\nreading ", nrow(ss_add), " additional sample(s): ",
          paste(need, collapse = ", "))
  add <- load_all_samples(ss_add, p)

  raw2 <- list(
    counts_probe = cbind_fill_union(list(res$raw$counts_probe, add$counts_probe)),
    meta = rbindlist(list(res$raw$meta, add$meta), use.names = TRUE, fill = TRUE))
  setkey(raw2$meta, cell)
  raw2$meta <- raw2$meta[colnames(raw2$counts_probe)]
  rm(add); gc(verbose = FALSE)

  p2 <- p
  p2$analysis_samples <- union(have, samples)
  p2$outlier_genes    <- p$outlier_genes %||% resolve_outliers(p)

  message("re-running QC and infection classification on ",
          format(ncol(raw2$counts_probe), big.mark = ","), " barcodes")
  qcres   <- stage_qc(raw2, p2)
  qc_kept <- qcres$qc[keep == TRUE]
  cls     <- classify_infection(qc_kept, p2, verbose = FALSE)

  .assert_calls_unchanged(res$classification$qc, cls$qc, have)

  out <- res
  out$params         <- p2
  out$raw            <- raw2
  out$qc             <- qcres$qc
  out$thresholds     <- qcres$thresholds
  out$qc_summary     <- qcres$summary
  out$classification <- cls

  if (verbose) {
    cat("\ncells per sample after QC, by infection call:\n")
    tab <- dcast(cls$qc[, .N, by = .(sample_id, timepoint, condition, infection)],
                 sample_id + timepoint + condition ~ infection,
                 value.var = "N", fill = 0)
    print(tab[order(timepoint, condition)])
  }
  out
}

In [ ]:
.make_pools <- function(n, k) sample(rep_len(seq_len(k), n))

pseudobulk_matrix <- function(res, tp, classes = BYSTANDER_CLASSES,
                              n_pools = 4, seed = 1234, p = res$params,
                              verbose = TRUE) {
  pr <- .timepoint_pair(res, tp, p)
  qc <- qc_with_offset_umi(res$classification$qc)
  cb <- qc[sample_id == pr$infected & infection %in% classes,
           .(cell, host_umi, sample_id = pr$infected, group = "bystander")]
  cc <- qc[sample_id == pr$control,
           .(cell, host_umi, sample_id = pr$control,  group = "control")]
  if (nrow(cb) < 2 * n_pools || nrow(cc) < 2 * n_pools)
    stop("too few cells at ", tp, " min (exposed ", nrow(cb), ", control ",
         nrow(cc), ") for ", n_pools, " pools per group")

  set.seed(seed)
  cb[, pool := .make_pools(.N, n_pools)]
  cc[, pool := .make_pools(.N, n_pools)]
  cells <- rbind(cb, cc)
  cells[, pool_id := factor(paste0(group, "_", pool),
                            levels = unique(paste0(group, "_", pool)))]

  cp <- res$raw$counts_probe[, cells$cell, drop = FALSE]
  P  <- Matrix::sparseMatrix(i = seq_len(nrow(cells)),
                             j = as.integer(cells$pool_id), x = 1,
                             dims = c(nrow(cells), nlevels(cells$pool_id)))
  M  <- cp %*% P                                    # probes x pools
  colnames(M) <- levels(cells$pool_id)
  M  <- as(as(M, "CsparseMatrix"), "dgCMatrix")
  G  <- aggregate_probes(M, "sum", p)               # genes x pools
  G  <- G[!is_phage(rownames(G), p), , drop = FALSE]
  G  <- drop_outlier_features(G, p, verbose = FALSE)

  cd <- cells[, .(n_cells = .N, host_umi = sum(host_umi)),
              by = .(pool_id, group, sample_id)]
  cd <- cd[match(colnames(G), as.character(pool_id))]
  cd[, group := factor(group, levels = c("control", "bystander"))]
  cd[, timepoint := tp]

  if (verbose)
    cat("\n[", tp, " min]  ", pr$infected, " exposed (n = ", nrow(cb),
        ") vs ", pr$control, " (n = ", nrow(cc), "):  ", nrow(G),
        " genes x ", ncol(G), " pools\n", sep = "")
  list(counts = G, colData = cd, pair = pr, cells = cells, timepoint = tp)
}

regulon_module_test_pb <- function(res, tps = c(15, 30, 45),
                                   regulons = REGULONS,
                                   classes = BYSTANDER_CLASSES, n_pools = 4,
                                   engine = c("DESeq2", "nb_glm"),
                                   size_factors = c("host_umi", "internal"),
                                   fit_type = "parametric",
                                   min_count = 10, seed = 1234,
                                   p = res$params, verbose = TRUE) {
  engine       <- match.arg(engine)
  size_factors <- match.arg(size_factors)
  if (engine == "DESeq2" && !requireNamespace("DESeq2", quietly = TRUE)) {
    message("DESeq2 is not installed -- falling back to nb_glm on the pools")
    engine <- "nb_glm"
  }
  mods <- names(regulons)

  out <- rbindlist(lapply(tps, function(tp) {
    pbm <- pseudobulk_matrix(res, tp, classes, n_pools, seed, p, verbose)
    G   <- pbm$counts; cd <- pbm$colData
    ex  <- cd$group == "bystander"

    gsl <- lapply(regulons, function(g)
      intersect(regulon_present(res, g, p, verbose = FALSE), rownames(G)))
    names(gsl) <- mods
    Mrows <- do.call(rbind, lapply(mods, function(m)
      Matrix::colSums(G[gsl[[m]], , drop = FALSE])))
    rownames(Mrows) <- paste0("MODULE_", mods)

    est <- if (engine == "DESeq2") {
      bg <- G[Matrix::rowSums(G) >= min_count, , drop = FALSE]
      A  <- rbind(as.matrix(bg), Mrows)
      storage.mode(A) <- "integer"
      cdf <- data.frame(group = cd$group); rownames(cdf) <- colnames(A)
      dds <- DESeq2::DESeqDataSetFromMatrix(A, cdf, ~ group)
      if (identical(size_factors, "host_umi"))
        DESeq2::sizeFactors(dds) <- cd$host_umi / exp(mean(log(cd$host_umi)))
      dds <- DESeq2::DESeq(dds, fitType = fit_type, quiet = TRUE)
      r   <- DESeq2::results(dds, contrast = c("group", "bystander", "control"))
      if (verbose)
        cat("  background: ", nrow(bg), " genes at >= ", min_count,
            " counts; ", sum(!is.na(r$padj) & r$padj < 0.05),
            " significant at padj < 0.05\n", sep = "")
      lapply(mods, function(m) {
        k <- paste0("MODULE_", m)
        list(log2FC = r[k, "log2FoldChange"],
             ci_lo  = r[k, "log2FoldChange"] - 1.96 * r[k, "lfcSE"],
             ci_hi  = r[k, "log2FoldChange"] + 1.96 * r[k, "lfcSE"],
             p = r[k, "pvalue"], model = "DESeq2 pseudobulk")
      })
    } else {
      lapply(mods, function(m) {
        z <- nb_glm(Mrows[paste0("MODULE_", m), ],
                    as.numeric(ex), cd$host_umi)
        list(log2FC = z$log2FC, ci_lo = z$ci_lo, ci_hi = z$ci_hi,
             p = z$p, model = paste0("pool ", z$model))
      })
    }
    names(est) <- mods

    rbindlist(lapply(mods, function(m) {
      y <- Mrows[paste0("MODULE_", m), ]
      e <- est[[m]]
      data.table(
        timepoint = tp, module = m, n_genes = length(gsl[[m]]),
        n_bystander = sum(pbm$cells$group == "bystander"),
        n_control   = sum(pbm$cells$group == "control"),
        n_pools = n_pools, total_UMIs = sum(y),
        rate_control   = 1000 * sum(y[!ex]) / sum(cd$host_umi[!ex]),
        rate_bystander = 1000 * sum(y[ ex]) / sum(cd$host_umi[ ex]),
        log2FC = e$log2FC, ci_lo = e$ci_lo, ci_hi = e$ci_hi,
        fold = 2^e$log2FC, p = e$p, model = e$model)
    }))
  }))

  out[, padj := p.adjust(p, "fdr")]     # modules x timepoints
  out[, stars := .sig_stars(padj)]
  setorder(out, timepoint, module)
  if (verbose) {
    cat("\n=== module fold change, pseudobulk (", out$model[1], ") ===\n", sep = "")
    print(out[, .(timepoint, module, n_genes, n_pools,
                  rate_control = round(rate_control, 2),
                  rate_bystander = round(rate_bystander, 2),
                  fold = round(fold, 2),
                  ci = paste0(round(2^ci_lo, 2), "-", round(2^ci_hi, 2)),
                  padj = signif(padj, 3), stars)])
  }
  out[]
}

In [ ]:
.group_gene_totals <- function(res, tp, classes = BYSTANDER_CLASSES,
                               p = res$params) {
  pr <- .timepoint_pair(res, tp, p)
  qc <- qc_with_offset_umi(res$classification$qc)
  d  <- rbind(
    qc[sample_id == pr$infected & infection %in% classes,
       .(cell, host_umi, grp = "exposed")],
    qc[sample_id == pr$control, .(cell, host_umi, grp = "control")])
  G <- aggregate_probes(res$raw$counts_probe[, d$cell, drop = FALSE], "sum", p)
  G <- G[!is_phage(rownames(G), p), , drop = FALSE]
  G <- drop_outlier_features(G, p, verbose = FALSE)
  ex <- d$grp == "exposed"
  list(exposed   = Matrix::rowSums(G[,  ex, drop = FALSE]),
       control   = Matrix::rowSums(G[, !ex, drop = FALSE]),
       H_exposed = sum(d$host_umi[ ex]),
       H_control = sum(d$host_umi[!ex]),
       n_exposed = sum(ex), n_control = sum(!ex), pair = pr)
}

.set_lfc <- function(z, genes) {
  e <- sum(z$exposed[genes]); k <- sum(z$control[genes])
  log2(((e + 0.5) / z$H_exposed) / ((k + 0.5) / z$H_control))
}

regulon_module_null <- function(res, mod_stats, regulons = REGULONS,
                                classes = BYSTANDER_CLASSES, n_null = 2000,
                                n_bins = 10, p = res$params, seed = 1234,
                                verbose = TRUE) {
  set.seed(seed)
  nul <- rbindlist(lapply(sort(unique(mod_stats$timepoint)), function(tp) {
    z   <- .group_gene_totals(res, tp, classes, p)
    tot <- z$exposed + z$control
    pool <- names(tot)[tot > 0]
    bin  <- setNames(cut(rank(tot[pool], ties.method = "first"),
                         n_bins, labels = FALSE), pool)
    by_bin <- split(pool, bin)

    rbindlist(lapply(names(regulons), function(m) {
      gs <- intersect(regulons[[m]], pool)
      if (!length(gs)) return(NULL)
      obs <- .set_lfc(z, gs)
      want <- table(bin[gs])
      null <- vapply(seq_len(n_null), function(i) {
        pick <- unlist(lapply(names(want), function(b)
          sample(by_bin[[b]], want[[b]])), use.names = FALSE)
        .set_lfc(z, pick)
      }, numeric(1))
      cen <- mean(null); sdv <- stats::sd(null)
      data.table(
        timepoint = tp, module = m, n_genes_tested = length(gs),
        lfc_obs = obs, null_mean = cen, null_sd = sdv,
        percentile = 100 * mean(null < obs),
        z_null = if (sdv > 0) (obs - cen) / sdv else NA_real_,
        p_null = (1 + sum(abs(null - cen) >= abs(obs - cen))) / (n_null + 1))
    }))
  }))
  nul[, padj_null := p.adjust(p_null, "fdr")]

  out <- merge(copy(mod_stats), nul, by = c("timepoint", "module"), all.x = TRUE)
  out[, stars_cell := stars]
  out[, stars := .sig_stars(padj_null)]
  setorder(out, timepoint, module)

  if (verbose) {
    cat("\n=== regulon vs ", n_null, " expression-matched random gene sets ===\n",
        sep = "")
    print(out[, .(timepoint, module,
                  fold = round(fold, 2),
                  null_fold = round(2^null_mean, 2),
                  pct = round(percentile, 1),
                  z = round(z_null, 1),
                  p_null = signif(p_null, 3),
                  cell_test = stars_cell, null_test = stars)])
    cat("\nnull_fold is what a typical matched gene set does between the same\n",
        "two libraries; pct is where the regulon falls in that distribution.\n",
        sep = "")
  }
  out[]
}

In [ ]:
if (!exists("GROUP_LABEL")) GROUP_LABEL <- "uninfected exposed"

.regulon_theme <- function(base_size = 12)
  ggplot2::theme_bw(base_size = base_size) +
  ggplot2::theme(
    panel.grid      = ggplot2::element_blank(),
    panel.border    = ggplot2::element_rect(colour = "grey20", linewidth = 0.6,
                                            fill = NA),
    plot.title      = ggplot2::element_text(face = "plain", hjust = 0.5,
                                            size = base_size * 1.2),
    plot.subtitle   = ggplot2::element_text(hjust = 0.5, colour = "grey30",
                                            size = base_size * 0.95),
    axis.text       = ggplot2::element_text(size = base_size * 1.0,
                                            colour = "grey15"),
    axis.title      = ggplot2::element_text(size = base_size * 1.05),
    legend.title    = ggplot2::element_blank(),
    legend.text     = ggplot2::element_text(size = base_size * 0.95),
    plot.background = ggplot2::element_rect(fill = "white", colour = NA))

.regulon_bar_panel <- function(mod_stats, regulon_labels = REGULON_LABELS,
                               base_size = 12, title = NULL, subtitle = NULL,
                               fill = NULL, outline = "grey15",
                               group_label = GROUP_LABEL, min_lfc = 0,
                               y_units = c("fold", "log2"), show_n = TRUE,
                               short_legend = TRUE, xlab = NULL, ylab = NULL,
                               bar_label = c("stars", "fold", "none")) {
  bar_label <- match.arg(bar_label)
  y_units   <- match.arg(y_units)
  disp <- if (isTRUE(short_legend))
    sub("\\s*regulon$", "", regulon_labels) else regulon_labels
  d <- data.table::copy(mod_stats)[is.finite(log2FC)]
  d[, tpf := factor(timepoint, levels = sort(unique(timepoint)))]
  d[, mod_lab := factor(unname(disp[module]),
                        levels = unname(disp[names(disp)]))]
  d[, lo := fifelse(is.finite(ci_lo), ci_lo, log2FC)]
  d[, hi := fifelse(is.finite(ci_hi), ci_hi, log2FC)]
  pad <- diff(range(c(d$lo, d$hi, 0), na.rm = TRUE)) * 0.06
  d[, star_y := fifelse(log2FC >= 0, hi + pad * 0.3, lo - pad * 0.3)]
  d[, star_vj := fifelse(log2FC >= 0, 0, 1)]
  d[, big := (lo > min_lfc) | (hi < -min_lfc)]
  d[, .lab := switch(bar_label,
                     stars = fifelse(big, stars, "ns"),
                     fold  = paste0(round(fold, 2), "\u00d7"),
                     none  = "")]

  mods <- names(regulon_labels)
  cols <- if (is.null(fill)) NULL else fill[mods]
  if (is.null(cols) || anyNA(cols) || is.null(names(fill)))
    cols <- grDevices::grey.colors(length(mods), start = 0.80, end = 0.30)
  cols <- setNames(unname(cols), unname(disp[mods]))
  dodge <- ggplot2::position_dodge(width = 0.72)

  yr <- range(c(d$lo, d$hi, d$star_y, 0), na.rm = TRUE)
  ysc <- if (y_units == "log2") ggplot2::scale_y_continuous() else {
    fb <- c(0.125, 0.25, 0.5, 1, 2, 4, 8, 16, 32)
    fb <- fb[log2(fb) >= yr[1] & log2(fb) <= yr[2]]
    ggplot2::scale_y_continuous(
      breaks = log2(fb),
      labels = sub("\\.?0+$", "", sprintf("%.3f", fb)))
  }
  .cap <- function(x) paste0(toupper(substring(x, 1, 1)), substring(x, 2))
  ylab <- ylab %||% paste0(.cap(group_label), " / control\n",
                           if (y_units == "log2") "(log2 fold change)"
                           else "fold change (log2 scale)")
  xlab <- xlab %||% "Time (minutes post infection)"

  xl <- ggplot2::waiver()
  if (isTRUE(show_n) && all(c("n_bystander", "n_control") %in% names(d))) {
    nn <- unique(d[, .(timepoint, n_bystander, n_control)])[order(timepoint)]
    xl <- paste0(nn$timepoint, "\n",
                 format(nn$n_bystander, big.mark = ",", trim = TRUE), " / ",
                 format(nn$n_control,   big.mark = ",", trim = TRUE))
  }

  ggplot2::ggplot(d, ggplot2::aes(tpf, log2FC, fill = mod_lab)) +
    ggplot2::geom_hline(yintercept = 0, linewidth = 0.5, colour = "grey30") +
    {if (min_lfc > 0) {
       gl <- c(-min_lfc, min_lfc); gl <- gl[gl >= yr[1] & gl <= yr[2]]
       if (length(gl)) ggplot2::geom_hline(yintercept = gl, linetype = 2,
         linewidth = 0.4, colour = "grey55") else NULL
     } else NULL} +
    ggplot2::geom_col(position = dodge, width = 0.64, colour = outline,
                      linewidth = 0.35) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = lo, ymax = hi), position = dodge,
                           width = 0.16, linewidth = 0.4, colour = "grey15") +
    {if (bar_label == "none") NULL else
      ggplot2::geom_text(ggplot2::aes(y = star_y, label = .lab, vjust = star_vj),
                         position = dodge, size = base_size * 0.3,
                         colour = "grey5")} +
    ggplot2::scale_fill_manual(values = cols) +
    ggplot2::scale_x_discrete(labels = xl) + ysc +
    ggplot2::expand_limits(y = c(min(d$lo, d$star_y) - pad,
                                 max(d$hi, d$star_y) + pad * 1.6)) +
    ggplot2::labs(title = title, subtitle = subtitle,
                  x = xlab, y = ylab) +
    .regulon_theme(base_size) +
    ggplot2::theme(legend.position = "top",
                   legend.key = ggplot2::element_rect(colour = outline,
                                                      linewidth = 0.35),
                   legend.text = ggplot2::element_text(
                     size = base_size * 0.9,
                     face = if (isTRUE(short_legend)) "italic" else "plain"),
                   axis.title.x = ggplot2::element_text(
                     margin = ggplot2::margin(t = 11)),
                   axis.title.y = ggplot2::element_text(
                     margin = ggplot2::margin(r = 6)),
                   panel.grid.major.y = ggplot2::element_line(colour = "grey90",
                                                              linewidth = 0.3))
}

fig_regulon_timecourse <- function(res, tps = c(15, 30, 45),
                                   regulons = REGULONS,
                                   regulon_labels = REGULON_LABELS,
                                   mod_stats = NULL,
                                   classes = BYSTANDER_CLASSES,
                                   base_size = 12, p = res$params,
                                   fill = NULL, bar_label = "stars",
                                   group_label = GROUP_LABEL, min_lfc = 0,
                                   y_units = "fold", show_n = TRUE,
                                   short_legend = TRUE,
                                   xlab = NULL, ylab = NULL,
                                   title = NULL, subtitle = NULL,
                                   name = "Fig2C_glp_mal_timecourse",
                                   width = NULL, height = NULL, dpi = 400,
                                   show = TRUE) {
  mods <- names(regulons)

  fig <- .regulon_bar_panel(
    mod_stats[timepoint %in% tps], regulon_labels, base_size,
    fill = fill, bar_label = bar_label, group_label = group_label,
    min_lfc = min_lfc, y_units = y_units, show_n = show_n,
    short_legend = short_legend, xlab = xlab, ylab = ylab,
    title = title, subtitle = subtitle)

  width  <- width  %||% (0.9 * length(tps) + 2.4)
  height <- height %||% (4.6 - 0.4 * is.null(title) - 0.2 * is.null(subtitle))
  save_paper_fig(fig, name, width, height, dpi, show = show)
}

In [ ]:
.rate_groups <- function(res, tp, classes = BYSTANDER_CLASSES, p = res$params) {
  pr <- .timepoint_pair(res, tp, p)
  qc <- qc_with_offset_umi(res$classification$qc)   # offset, not a filter
  list(exposed  = qc[sample_id == pr$infected &  infection %in% classes],
       control  = qc[sample_id == pr$control],
       infected = qc[sample_id == pr$infected & !infection %in% classes])
}

regulon_rate_table <- function(res, tps = c(15, 30, 45), regulons = REGULONS,
                               classes = BYSTANDER_CLASSES,
                               groups = c("exposed", "control"),
                               mode = c("rate", "relative"),
                               n_boot = 500, seed = 1234, p = res$params) {
  mode <- match.arg(mode)
  set.seed(seed)
  sel <- lapply(regulons, function(g) .probe_rows(res, g, p))

  raw <- rbindlist(lapply(tps, function(tp) {
    gl <- .rate_groups(res, tp, classes, p)
    rbindlist(lapply(intersect(groups, names(gl)), function(gn) {
      z <- gl[[gn]]
      if (!nrow(z)) return(NULL)
      rbindlist(lapply(names(regulons), function(m) {
        data.table(timepoint = tp, group = gn, module = m,
                   counts = as.numeric(Matrix::colSums(
                     res$raw$counts_probe[sel[[m]], z$cell, drop = FALSE])),
                   host_umi = z$host_umi)
      }))
    }))
  }))

  pooled <- function(d) 1000 * sum(d$counts) / sum(d$host_umi)
  obs <- raw[, .(n_cells = .N, rate = pooled(.SD)), by = .(timepoint, group, module)]

  boots <- raw[, {
    v <- vapply(seq_len(n_boot), function(i) {
      j <- sample.int(.N, .N, replace = TRUE)
      1000 * sum(counts[j]) / sum(host_umi[j])
    }, numeric(1))
    .(b = list(v))
  }, by = .(timepoint, group, module)]
  obs <- merge(obs, boots, by = c("timepoint", "group", "module"))

  if (mode == "rate") {
    obs[, `:=`(value = rate,
               lo = vapply(b, stats::quantile, numeric(1), 0.025, names = FALSE),
               hi = vapply(b, stats::quantile, numeric(1), 0.975, names = FALSE))]
  } else {
    base <- min(tps)
    obs[, idx := .I]
    for (i in obs$idx) {
      g <- obs$group[i]; m <- obs$module[i]
      j <- obs[group == g & module == m & timepoint == base, which = TRUE]
      r <- obs$b[[i]] / obs$b[[j]]                # paired across bootstrap reps
      set(obs, i, "value", obs$rate[i] / obs$rate[j])
      set(obs, i, "lo", as.numeric(stats::quantile(r, 0.025)))
      set(obs, i, "hi", as.numeric(stats::quantile(r, 0.975)))
    }
    obs[, idx := NULL]
  }
  obs[, b := NULL]
  obs[, group := factor(group, levels = c("exposed", "infected", "control"))]
  setorder(obs, module, timepoint, group)
  obs[]
}

fig_regulon_rates <- function(res, tps = c(15, 30, 45), regulons = REGULONS,
                              regulon_labels = REGULON_LABELS,
                              rates = NULL, classes = BYSTANDER_CLASSES,
                              groups = c("exposed", "control"),
                              mode = c("rate", "relative"),
                              n_boot = 500, base_size = 12, p = res$params,
                              short_labels = TRUE, xlab = NULL, ylab = NULL,
                              name = "Fig2C_rates", width = NULL,
                              height = 3.9, dpi = 400, show = TRUE) {
  mode  <- match.arg(mode)
  d <- data.table::copy(rates %||%
    regulon_rate_table(res, tps, regulons, classes, groups, mode, n_boot, p = p))
  d[, tpf := factor(timepoint, levels = sort(unique(timepoint)))]
  disp <- if (isTRUE(short_labels))
    sub("\\s*regulon$", "", regulon_labels) else regulon_labels
  d[, mod_lab := factor(unname(disp[module]), levels = unname(disp[names(disp)]))]
  gl <- c(exposed = "Exposed-uninfected", infected = "Infected",
          control = "Control")
  d[, group := factor(as.character(group), levels = names(gl)[names(gl) %in% group])]
  d[, grp_lab := factor(unname(gl[as.character(group)]),
                        levels = unname(gl[levels(group)]))]
  grey_of <- c(exposed = "#dcdcdc", infected = "#9e9e9e", control = "#4f4f4f")
  cols <- setNames(unname(grey_of[levels(d$group)]),
                   levels(d$grp_lab))
  dodge <- ggplot2::position_dodge(width = 0.72)

  g <- ggplot2::ggplot(d, ggplot2::aes(tpf, value, fill = grp_lab)) +
    ggplot2::geom_col(position = dodge, width = 0.64, colour = "grey15",
                      linewidth = 0.35) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = lo, ymax = hi), position = dodge,
                           width = 0.16, linewidth = 0.4, colour = "grey15") +
    ggplot2::scale_fill_manual(values = cols) +
    ggplot2::facet_wrap(~ mod_lab, nrow = 1, scales = "free_y") +
    ggplot2::scale_y_continuous(expand = ggplot2::expansion(mult = c(0, 0.08))) +
    ggplot2::labs(x = xlab %||% "Time (minutes post infection)",
                  y = ylab %||% if (mode == "rate")
                        "Transcripts per 1,000 host UMIs" else
                        paste0("Expression relative to ", min(tps), " min")) +
    .regulon_theme(base_size) +
    ggplot2::theme(legend.position = "top",
                   legend.key = ggplot2::element_rect(colour = "grey15",
                                                      linewidth = 0.35),
                   strip.background = ggplot2::element_blank(),
                   strip.text = ggplot2::element_text(
                     hjust = 0, size = base_size * 1.0,
                     face = if (isTRUE(short_labels)) "italic" else "plain",
                     margin = ggplot2::margin(b = 4)),
                   panel.spacing = grid::unit(14, "pt"),
                   axis.title.x = ggplot2::element_text(
                     margin = ggplot2::margin(t = 9)),
                   axis.title.y = ggplot2::element_text(
                     margin = ggplot2::margin(r = 6)),
                   panel.grid.major.y = ggplot2::element_line(colour = "grey91",
                                                              linewidth = 0.3))
  if (mode == "relative")
    g <- g + ggplot2::geom_hline(yintercept = 1, linewidth = 0.4,
                                 colour = "grey30", linetype = 2)
  width <- width %||% (2.5 * length(regulons) +
                       0.25 * length(tps) * nlevels(d$grp_lab))
  save_paper_fig(g, name, width, height, dpi, show = show)
}

In [ ]:
F_TRANSFER_MAP <- c(
  LELOBEKK_04440 = "traM", LELOBEKK_04441 = "traJ", LELOBEKK_04442 = "traY",
  LELOBEKK_04443 = "traA", LELOBEKK_04444 = "traL", LELOBEKK_04445 = "traE",
  LELOBEKK_04446 = "traK", LELOBEKK_04447 = "traB", LELOBEKK_04448 = "traP",
  LELOBEKK_04449 = "trbD", LELOBEKK_04450 = "trbG", LELOBEKK_04451 = "traV",
  LELOBEKK_04453 = "traC", LELOBEKK_04454 = "trbI", LELOBEKK_04455 = "traW",
  LELOBEKK_04456 = "traU", LELOBEKK_04457 = "trbC", LELOBEKK_04458 = "traN",
  LELOBEKK_04459 = "trbE", LELOBEKK_04460 = "traF", LELOBEKK_04461 = "trbA",
  LELOBEKK_04463 = "traQ", LELOBEKK_04464 = "trbB", LELOBEKK_04465 = "trbJ",
  LELOBEKK_04466 = "trbF", LELOBEKK_04467 = "traH", LELOBEKK_04468 = "traG",
  LELOBEKK_04469 = "traS", LELOBEKK_04470 = "traT", LELOBEKK_04471 = "traD",
  LELOBEKK_04474 = "traI", LELOBEKK_04475 = "traX", LELOBEKK_04476 = "finO")

F_PANEL_ALIASES <- c(LELOBEKK_ylpA = "traT")

NONINF_CLASSES <- BYSTANDER_CLASSES

.resolve_f_genes <- function(res, map = F_TRANSFER_MAP, p = res$params,
                             allow_bare = TRUE, aliases = F_PANEL_ALIASES,
                             outlier_scope = c("panel_gene", "symbol"),
                             verbose = TRUE) {
  outlier_scope <- match.arg(outlier_scope)
  feats <- rownames(res$raw$counts_probe)
  panel <- sort(unique(probe_to_gene(feats[!is_phage(feats, p)], p)))
  og    <- p$outlier_genes %||% character(0)

  rows <- data.table::rbindlist(lapply(seq_along(map), function(k) {
    tag  <- names(map)[k]; sym <- unname(map[k])
    cand <- unique(c(tag, paste0("LELOBEKK_", sym), if (allow_bare) sym,
                     names(aliases)[aliases == sym]))
    hit  <- panel[tolower(panel) %in% tolower(cand)]
    if (!length(hit)) return(NULL)
    data.table::data.table(panel_gene = hit, symbol = sym, locus = tag,
                           how = data.table::fifelse(
                             tolower(hit) == tolower(tag), "locus tag",
                             data.table::fifelse(
                               tolower(hit) == tolower(paste0("LELOBEKK_", sym)),
                               "prefixed symbol",
                               data.table::fifelse(
                                 tolower(hit) %in% tolower(names(aliases)),
                                 "alias", "bare symbol"))))
  }))

  dropped <- character(0)
  if (nrow(rows) && length(og)) {
    hit <- tolower(rows$panel_gene) %in% tolower(og)
    if (identical(outlier_scope, "symbol")) {
      bad <- unique(rows$symbol[hit])
      hit <- rows$symbol %in% bad
    }
    dropped <- rows$panel_gene[hit]
    rows    <- rows[!hit]
  }
  missing <- setdiff(unname(map), if (nrow(rows)) rows$symbol else character(0))

  if (verbose) {
    cat("\n--- F transfer region on the probe panel ---\n")
    cat("  ", nrow(rows), " panel gene(s) -> ",
        if (nrow(rows)) data.table::uniqueN(rows$symbol) else 0, " of ",
        length(map), " symbol(s); ",
        if (nrow(rows)) length(.probe_rows(res, rows$panel_gene, p)) else 0,
        " probe(s)\n", sep = "")
    if (nrow(rows)) {
      print(rows[order(symbol), .(symbol, panel_gene, how)])
      dup <- rows[, .N, by = symbol][N > 1]
      if (nrow(dup))
        message("  summed as one gene (panel annotates them twice): ",
                paste(dup$symbol, collapse = ", "))
    }
    if (length(missing))
      cat("  NOT on the panel (", length(missing), "): ",
          paste(missing, collapse = ", "), "\n", sep = "")
    if (length(dropped))
      cat("  dropped as outlier gene(s): ", paste(dropped, collapse = ", "),
          if (identical(outlier_scope, "symbol"))
            "\n    (every spelling of a flagged symbol)"
          else
            "\n    (only these exact names; other spellings of the same symbol remain)",
          "\n", sep = "")
  }
  if (!nrow(rows))
    stop("no gene of this set is on the probe panel.\n",
         "  Panel names look like: ",
         paste(utils::head(panel, 6), collapse = ", "),
         "\n  Edit F_TRANSFER_MAP, or pass gmap = <your own table>.")
  rows[]
}

plasmid_gene_counts <- function(res, sid, gmap, p = res$params,
                                noninf_classes = NONINF_CLASSES) {
  qc <- qc_with_offset_umi(res$classification$qc)[as.character(sample_id) == sid]
  if (!nrow(qc))
    stop("no cells for '", sid, "'. Read: ",
         paste(unique(as.character(res$classification$qc$sample_id)),
               collapse = ", "))
  rows <- .probe_rows(res, gmap$panel_gene, p)   # outlier genes already gone
  if (!length(rows)) stop("no probe rows for this gene set in '", sid, "'")

  G <- aggregate_probes(res$raw$counts_probe[rows, qc$cell, drop = FALSE],
                        "sum", p)

  sym <- gmap$symbol[match(tolower(rownames(G)), tolower(gmap$panel_gene))]
  idx <- split(seq_len(nrow(G)), factor(sym, levels = unique(gmap$symbol)))
  idx <- idx[lengths(idx) > 0]
  Gs  <- t(vapply(idx, function(ii) Matrix::colSums(G[ii, , drop = FALSE]),
                  numeric(ncol(G))))
  rownames(Gs) <- names(idx); colnames(Gs) <- colnames(G)

  list(counts = Gs, genes = rownames(Gs), cells = qc$cell,
       host   = qc$host_umi,                  # probe-level host total
       noninf = qc$infection %in% noninf_classes)
}

plasmid_gene_totals <- function(res, samples, gmap, p = res$params,
                                verbose = TRUE) {
  pre <- data.table::rbindlist(lapply(samples, function(sid) {
    qc   <- qc_with_offset_umi(res$classification$qc)[as.character(sample_id) == sid]
    rows <- .probe_rows(res, gmap$panel_gene, p)
    G    <- aggregate_probes(res$raw$counts_probe[rows, qc$cell, drop = FALSE],
                             "sum", p)
    data.table::data.table(
      sample = sid, panel_gene = rownames(G),
      UMIs = as.numeric(Matrix::rowSums(G)),
      cells_detected = as.integer(Matrix::rowSums(G > 0)),
      n_cells = ncol(G))
  }))
  pre <- merge(pre, gmap[, .(panel_gene, symbol, how)],
               by = "panel_gene", all.x = TRUE)
  data.table::setcolorder(pre, c("sample", "symbol", "panel_gene", "how"))
  data.table::setorder(pre, sample, symbol, panel_gene)

  post <- pre[, .(n_panel_genes = .N,
                  from = paste(panel_gene, collapse = " + "),
                  UMIs = sum(UMIs),
                  cells_detected = max(cells_detected)),
              by = .(sample, symbol)]
  data.table::setorder(post, sample, -UMIs)

  if (verbose) {
    cat("\n--- BEFORE the symbol merge: UMIs per PANEL gene ---\n")
    print(pre[, .(sample, symbol, panel_gene, how, UMIs = round(UMIs),
                  cells_detected)])
    cat("\n--- AFTER the symbol merge: UMIs per SYMBOL (what is plotted) ---\n")
    print(post[, .(sample, symbol, n_panel_genes, from, UMIs = round(UMIs),
                   cells_detected)])
    merged <- unique(post[n_panel_genes > 1, symbol])
    if (length(merged))
      message("  summed from more than one panel gene: ",
              paste(merged, collapse = ", "),
              "\n  Check these are duplicate annotations of one locus and not ",
              "two distinct features -- the sum double-counts in the first case.")
  }
  invisible(list(before = pre[], after = post[]))
}

plasmid_module_test <- function(res, samples, gmap, p = res$params,
                                noninf_classes = NONINF_CLASSES) {
  data.table::rbindlist(lapply(samples, function(sid) {
    z   <- plasmid_gene_counts(res, sid, gmap, p, noninf_classes)
    y   <- colSums(z$counts)
    fit <- nb_glm(y, as.integer(z$noninf), z$host)   # grp = 1 -> non-infected
    data.table::data.table(
      sample = sid,
      n_infected = sum(!z$noninf), n_noninfected = sum(z$noninf),
      mean_infected_per1000 = 1000 * sum(y[!z$noninf]) / sum(z$host[!z$noninf]),
      mean_noninf_per1000   = 1000 * sum(y[ z$noninf]) / sum(z$host[ z$noninf]),
      log2FC = fit$log2FC, ci_lo = fit$ci_lo, ci_hi = fit$ci_hi,
      p = fit$p, model = fit$model, fold = 2^fit$log2FC)
  }))
}

plasmid_per_gene_test <- function(res, samples, gmap, p = res$params,
                                  noninf_classes = NONINF_CLASSES) {
  out <- data.table::rbindlist(lapply(samples, function(sid) {
    z <- plasmid_gene_counts(res, sid, gmap, p, noninf_classes)
    data.table::rbindlist(lapply(z$genes, function(g) {
      y   <- as.numeric(z$counts[g, ])
      fit <- nb_glm(y, as.integer(z$noninf), z$host)
      data.table::data.table(
        sample = sid, gene = g, total_UMIs = sum(y),
        n_cells_detected = sum(y > 0),
        mean_infected_per1000 = 1000 * sum(y[!z$noninf]) / sum(z$host[!z$noninf]),
        mean_noninf_per1000   = 1000 * sum(y[ z$noninf]) / sum(z$host[ z$noninf]),
        log2FC = fit$log2FC, p = fit$p, model = fit$model)
    }))
  }))
  out[, fold := 2^log2FC]
  out[, padj := stats::p.adjust(p, "BH"), by = sample]
  out[]
}

fig_plasmid_transfer <- function(res, samples = c("Inf_30m", "Inf_45m"),
                                 map = F_TRANSFER_MAP, gmap = NULL,
                                 p = res$params,
                                 noninf_classes = NONINF_CLASSES,
                                 mod_stats = NULL, gene_stats = NULL,
                                 report_counts = TRUE,
                                 min_umi = 150,
                                 fold_limits = c(0.25, 1.5),
                                 base_size = 15,
                                 cols = c(infected = "#FF7F0E",
                                          noninf   = "#1F77B4"),
                                 tp_cols = c("#cfcfcf", "#636363"),
                                 noninf_label = "Exposed-uninfected",
                                 module_label = "F-pilus transfer operon",
                                 width = NULL, height = NULL, name = NULL,
                                 formats = NULL, dpi = 400, show = TRUE) {

  if (!requireNamespace("patchwork", quietly = TRUE))
    stop("'patchwork' is required to combine the panels")

  if (is.null(gmap)) gmap <- .resolve_f_genes(res, map, p)
  if (report_counts) plasmid_gene_totals(res, samples, gmap, p)
  if (is.null(mod_stats))
    mod_stats  <- plasmid_module_test(res, samples, gmap, p, noninf_classes)
  if (is.null(gene_stats))
    gene_stats <- plasmid_per_gene_test(res, samples, gmap, p, noninf_classes)

  .tp_of <- function(s) {
    n <- regmatches(s, regexpr("[0-9]+", s))
    if (length(n) && nzchar(n)) .tp_label(n) else s
  }

  th <- paper_theme(base_size, "labels") +
    ggplot2::theme(
      legend.title    = ggplot2::element_blank(),
      legend.position = "bottom",
      legend.margin   = ggplot2::margin(t = -2),
      legend.key.size = grid::unit(base_size * 0.9, "pt"),
      legend.text     = ggplot2::element_text(size = base_size * 0.85),
      plot.title      = ggplot2::element_text(face = "plain", hjust = 0.5,
                          size = base_size * 1.1,
                          margin = ggplot2::margin(b = 5)))

  panels <- lapply(seq_along(samples), function(i) {
    sid <- samples[i]
    z   <- plasmid_gene_counts(res, sid, gmap, p, noninf_classes)
    v   <- 1000 * colSums(z$counts) / z$host
    d   <- data.table::data.table(
      expr = as.numeric(v),
      grp  = factor(ifelse(z$noninf, noninf_label, "Infected"),
                    levels = c("Infected", noninf_label)))

    st <- mod_stats[sample == sid]
    if (nrow(st) != 1)
      stop("expected one row of mod_stats for '", sid, "', got ", nrow(st))

    ytop <- unname(stats::quantile(log10(d$expr + 1), 0.96, na.rm = TRUE)) + 0.12
    lab  <- c(paste0("n = ", sum(!z$noninf)),
              paste0("n = ", sum(z$noninf)))

    g <- ggplot2::ggplot(d, ggplot2::aes(grp, log10(expr + 1), fill = grp)) +
      ggplot2::geom_violin(width = 0.85, alpha = 0.35, colour = NA,
                           scale = "width") +
      ggplot2::geom_boxplot(width = 0.3, outlier.shape = NA, alpha = 0.92,
                            colour = "black", linewidth = 0.4, fatten = 2.2,
                            show.legend = FALSE) +
      ggplot2::scale_fill_manual(
        values = stats::setNames(c(cols[["infected"]], cols[["noninf"]]),
                                 c("Infected", noninf_label))) +
      ggplot2::scale_x_discrete(labels = lab) +
      ggplot2::scale_y_continuous(breaks = 0:2, labels = c("0", "10", "100")) +
      ggplot2::annotate("segment", x = 1, xend = 2, y = ytop + 0.05,
                        yend = ytop + 0.05, linewidth = 0.4) +
      ggplot2::annotate("segment", x = 1, xend = 1, y = ytop,
                        yend = ytop + 0.05, linewidth = 0.4) +
      ggplot2::annotate("segment", x = 2, xend = 2, y = ytop,
                        yend = ytop + 0.05, linewidth = 0.4) +
      ggplot2::annotate("text", x = 1.5, y = ytop + 0.09,
                        label = paste0(round(st$fold, 2), "\u00d7  ",
                                       .sig_stars(st$p)),
                        size = base_size * 0.34, vjust = 0) +
      ggplot2::labs(title = .tp_of(sid), x = NULL,
                    y = if (i == 1)
                          paste0(module_label,
                                 " expression\n(Transcripts per 1,000 host UMIs)")
                        else NULL) +
      th
    if (i != 1)
      g <- g + ggplot2::theme(axis.text.y  = ggplot2::element_blank(),
                              axis.ticks.y = ggplot2::element_blank())
    g
  })

  gd <- data.table::copy(gene_stats)[is.finite(log2FC)]
  n_all <- data.table::uniqueN(gene_stats$gene)
  keep  <- gd[, .(tot = max(total_UMIs)), by = gene][tot >= min_umi, gene]
  gd    <- gd[gene %in% keep]
  if (!nrow(gd)) stop("no gene reaches min_umi = ", min_umi)
  cat("  well-detected genes: ", data.table::uniqueN(gd$gene), " of ", n_all,
      " at >= ", min_umi, " UMIs in at least one timepoint\n", sep = "")

  fold_limits <- sort(fold_limits)
  gd[, fold_plot := pmin(pmax(fold, fold_limits[1]), fold_limits[2])]
  gd[, x := log2(fold_plot)]
  gd[, clipped := fold < fold_limits[1] | fold > fold_limits[2]]

  if (sum(gd$clipped))
    message("  ", sum(gd$clipped), " point(s) outside [", fold_limits[1], ", ",
            fold_limits[2], "] clamped to the axis edge, drawn as open diamonds:\n    ",
            paste(gd[clipped == TRUE,
                     paste0(gene, " (", sample, ", fold ", signif(fold, 2),
                            ", ", total_UMIs, " UMIs)")], collapse = "\n    "))

  ord <- gd[sample == samples[length(samples)]][order(x), gene]
  ord <- c(ord, setdiff(unique(gd$gene), ord))
  gd[, gene := factor(gene, levels = ord)]
  gd[, tp := factor(vapply(sample, .tp_of, "", USE.NAMES = FALSE),
                    levels = vapply(samples, .tp_of, "", USE.NAMES = FALSE))]

  brks <- c(0.125, 0.25, 0.5, 0.75, 1, 1.5, 2, 4, 8)
  brks <- brks[brks >= fold_limits[1] & brks <= fold_limits[2]]
  fmt  <- function(z) sub("\\.?0+$", "", sprintf("%.3f", z))

  pC <- ggplot2::ggplot(gd, ggplot2::aes(x, gene, fill = tp)) +
    ggplot2::geom_vline(xintercept = 0, linetype = 2, linewidth = 0.5,
                        colour = "grey20") +
    ggplot2::geom_point(size = 3.4, shape = 21, colour = "grey25", stroke = 0.5,
                        alpha = 0.9) +
    ggplot2::scale_fill_manual(
      values = stats::setNames(tp_cols[seq_along(samples)], levels(gd$tp))) +
    ggplot2::scale_x_continuous(breaks = log2(brks), labels = fmt(brks),
                                limits = log2(fold_limits),
                                expand = ggplot2::expansion(mult = 0.02)) +
    ggplot2::labs(title = NULL, y = NULL,
                  x = paste0("Fold change\n(", noninf_label,
                             " \u00f7 infected)")) +
    th + ggplot2::theme(
      panel.grid.major.x = ggplot2::element_line(colour = "grey90",
                                                 linewidth = 0.3),
      axis.ticks.y = ggplot2::element_blank(),
      axis.text.y  = ggplot2::element_text(size = base_size * 0.8),
      plot.title   = ggplot2::element_text(face = "plain", hjust = 0.5,
                       size = base_size * 1.1,
                       margin = ggplot2::margin(b = 5))) +
    ggplot2::guides(fill = ggplot2::guide_legend(
      nrow = 1, override.aes = list(shape = 22, size = 4.5, stroke = 0.4)))

  cat("\nmodule test (exposed-uninfected vs infected, NB GLM, host-UMI offset):\n")
  print(mod_stats[, .(sample, n_infected, n_noninfected,
                      per1000_infected = round(mean_infected_per1000, 2),
                      per1000_noninf   = round(mean_noninf_per1000, 2),
                      fold = round(fold, 2), p = signif(p, 3), model)])

  left <- patchwork::wrap_plots(panels, nrow = 1) +
    patchwork::plot_layout(guides = "collect") &
    ggplot2::theme(legend.position = "bottom")
  fig <- patchwork::wrap_plots(left, pC, nrow = 1,
                               widths = c(length(panels), 1.25))

  width  <- width  %||% (3.4 * length(panels) + 4.6)
  height <- height %||% 5.8
  name   <- name   %||% "Fig2B_transfer_operon"
  save_paper_fig(fig, name, width, height, dpi,
                 formats = formats %||% FIG_FORMATS, show = show)
  invisible(list(figure = fig, module = mod_stats, genes = gene_stats,
                 gmap = gmap))
}

In [ ]:
log1p_trans <- function() {
  nt <- if ("new_transform" %in% getNamespaceExports("scales"))
          scales::new_transform else scales::trans_new
  nt("log1p", transform = log1p, inverse = expm1, domain = c(0, Inf))
}

operon_rate <- function(res, sid, genes, p = res$params, per = 1000) {
  a <- res$per_sample[[sid]]
  if (is.null(a))
    stop("'", sid, "' has no per-sample analysis (no UMAP to draw on).\n",
         "  res$per_sample holds: ", paste(names(res$per_sample), collapse = ", "),
         "\n  Add it to RUN_SAMPLES, or re-run with ",
         "analysis_samples = c(...) and cached_raw = raw_exp1.")
  cells <- colnames(a$obj)

  cp    <- res$raw$counts_probe
  feats <- rownames(cp)
  sel   <- !is_phage(feats, p) &
           tolower(probe_to_gene(feats, p)) %in% tolower(genes)
  if (!any(sel))
    stop("none of these genes are on the probe panel: ",
         paste(utils::head(genes, 8), collapse = ", "))

  y   <- Matrix::colSums(cp[sel, cells, drop = FALSE])
  qc  <- qc_with_offset_umi(res$classification$qc)
  den <- qc$host_umi[match(cells, qc$cell)]
  if (anyNA(den)) stop(sum(is.na(den)), " cells of ", sid, " are not in the QC table")

  r <- per * y / den
  names(r) <- cells
  r
}

.resolve_gene_set <- function(x, res, p, exclude = NULL) {
  feats    <- rownames(res$raw$counts_probe)
  on_panel <- sort(unique(probe_to_gene(feats[!is_phage(feats, p)], p)))
  g <- if (length(x) == 1) on_panel[grepl(x, on_panel, ignore.case = TRUE)] else x
  drop <- c(p$outlier_genes %||% character(0),
            exclude %||% (if (exists("REGULON_EXCLUDE")) REGULON_EXCLUDE
                          else character(0)))
  g <- g[!tolower(g) %in% tolower(drop)]
  list(genes   = sort(unique(intersect(g, on_panel))),
       missing = if (length(x) > 1) setdiff(x, on_panel) else character(0))
}

fig_operon_grid <- function(res,
                            samples  = c("Ctrl_45m", "Inf_45m"),
                            operons  = list("fim operon" = "^fim",
                                            "srl operon" = "^srl"),
                            p = res$params, per = 1000,
                            include_clusters = TRUE,
                            row_labels = NULL,
                            rate_limits = NULL, clip = 0.99,
                            breaks = c(0, 1, 2, 5, 10, 20),
                            trans = log1p_trans(),
                            palette = "inferno", direction = 1,
                            legend_title = "Expression\n(per 1,000 host UMIs)",
                            base_size = 13, axis_style = "labels",
                            point_size = NULL, point_alpha = 0.9,
                            rasterise = TRUE, dpi = 400, formats = NULL,
                            width = NULL, height = NULL, name = NULL,
                            show = TRUE) {

  if (!requireNamespace("patchwork", quietly = TRUE))
    stop("'patchwork' is required to build the grid")
  miss <- setdiff(samples, names(res$per_sample))
  if (length(miss))
    stop("no per-sample analysis for: ", paste(miss, collapse = ", "),
         "\n  res$per_sample holds: ",
         paste(names(res$per_sample), collapse = ", "),
         "\n  The default run clusters Inf_45m only. Re-run with",
         "\n    p <- params_exp1; p$analysis_samples <- c(",
         paste0('"', samples, '"', collapse = ", "), ")",
         "\n    res_op <- run_pipeline(p, cached_raw = raw_exp1)")

  gs <- lapply(operons, .resolve_gene_set, res = res, p = p)
  names(gs) <- names(operons)
  cat("\n--- operon gene sets ---\n")
  for (nm in names(gs)) {
    if (!length(gs[[nm]]$genes))
      stop("'", nm, "' matched no gene on the probe panel")
    cat("  ", nm, ": ", length(gs[[nm]]$genes), " gene(s) -- ",
        paste(gs[[nm]]$genes, collapse = ", "), "\n", sep = "")
    if (length(gs[[nm]]$missing))
      message("    not on the panel, ignored: ",
              paste(gs[[nm]]$missing, collapse = ", "))
  }

  tabs <- lapply(samples, function(sid) {
    ex <- lapply(gs, function(g) operon_rate(res, sid, g$genes, p, per))
    names(ex) <- names(gs)
    umap_table(res, sid, extra = ex)
  })
  names(tabs) <- samples

  if (is.null(rate_limits)) {
    allv <- unlist(lapply(tabs, function(d)
      unlist(lapply(names(gs), function(nm) as.numeric(d[[nm]])))))
    hi <- as.numeric(stats::quantile(allv, clip, na.rm = TRUE))
    rate_limits <- c(0, max(hi, 1))
    cat("\ncolour scale: 0 to ", signif(rate_limits[2], 3),
        " per ", per, " host UMIs (", clip * 100, "th percentile of ",
        format(length(allv), big.mark = ","), " cell x operon values; observed max ",
        signif(max(allv, na.rm = TRUE), 3), ")\n", sep = "")
  }
  brk <- breaks[breaks >= rate_limits[1] & breaks <= rate_limits[2]]
  if (length(brk) < 2) brk <- pretty(rate_limits, n = 4)

  if (is.null(row_labels)) {
    lab <- vapply(samples, pretty_sample, character(1), USE.NAMES = FALSE)
    if (length(unique(sub("^[^,]*,?\\s*", "", lab))) == 1)
      lab <- sub(",.*$", "", lab)
    row_labels <- lab
  }

  cols <- c(if (!is.null(row_labels)) "label",
            if (include_clusters) "cluster", names(gs))
  panels <- list()
  for (ri in seq_along(samples)) {
    d    <- tabs[[ri]]
    lims <- .axis_limits(d)              # each row gets its own UMAP extent
    for (cj in cols) {
      panels <- c(panels, list(
        if (identical(cj, "label")) {
          row_label_panel(row_labels[ri], base_size)
        } else if (identical(cj, "cluster")) {
          panel_umap(d, "cluster", title = if (ri == 1) "Clusters" else NULL,
                     lims = lims, cluster_labels = TRUE, base_size = base_size,
                     axis_style = axis_style, point_size = point_size,
                     point_alpha = point_alpha, rasterise = rasterise, dpi = dpi)
        } else {
          panel_umap(d, "value", value_col = cj,
                     title = if (ri == 1) cj else NULL,
                     lims = lims, limits = rate_limits, palette = palette,
                     direction = direction, legend_title = legend_title,
                     breaks = brk,
                     labels = function(z) format(z, trim = TRUE,
                                                 scientific = FALSE),
                     trans = trans, base_size = base_size,
                     axis_style = axis_style, point_size = point_size,
                     point_alpha = point_alpha, rasterise = rasterise, dpi = dpi)
        }))
    }
  }

  w <- ifelse(cols == "label", 0.10, 1)
  fig <- patchwork::wrap_plots(panels, ncol = length(cols), widths = w,
                               guides = "collect")

  n_panel_cols <- sum(cols != "label")
  width  <- width  %||% (0.5 + 2.7 * n_panel_cols + 1.4)
  height <- height %||% (0.4 + 2.6 * length(samples))
  name   <- name   %||% paste0("SuppFig_operon_grid_",
                               paste(samples, collapse = "_vs_"))
  save_paper_fig(fig, name, width, height, dpi,
                 formats = formats %||% FIG_FORMATS, show = show)
}

In [ ]:
fig_ms2_strand_hist <- function(res, samples = NULL, p = res$params,
                                strand_map = c("(+) strand" = "rev",
                                               "(-) strand" = "fwd"),
                                bins = 42, base_size = 15, free_y = TRUE,
                                alpha = 0.62, formats = "png",
                                colours = c(infected = "#FF7F0E",
                                            control  = "#1F77B4"),
                                cond_labels = c(infected = "Infected",
                                                control  = "Control"),
                                width = NULL, height = NULL, name = NULL,
                                show = TRUE) {

  qc <- data.table::as.data.table(res$classification$qc)
  for (nm in c("cell", "sample_id", "condition", "timepoint"))
    if (!nm %in% names(qc)) stop("'", nm, "' missing from res$classification$qc")

  have <- unique(as.character(qc$sample_id))
  if (is.null(samples)) samples <- have
  miss <- setdiff(samples, have)
  if (length(miss))
    stop("not in this result: ", paste(miss, collapse = ", "),
         "\n  available: ", paste(have, collapse = ", "),
         "\n  For all six exp1 samples use res_tc (what ensure_samples_read() ",
         "returns), not res_exp1.")
  qc <- qc[as.character(sample_id) %in% samples]

  cp <- res$raw$counts_probe
  if (is.null(cp)) stop("res$raw$counts_probe is missing -- keep the raw object")
  gone <- setdiff(qc$cell, colnames(cp))
  if (length(gone))
    stop(length(gone), " cells in the QC table are not in the probe matrix")

  ph <- rownames(cp)[is_phage(rownames(cp), p)]
  if (!length(ph)) stop("no features match phage_prefix '", p$phage_prefix, "'")

  sel <- lapply(strand_map, function(rx)
    grep(rx, ph, value = TRUE, ignore.case = isTRUE(p$phage_ignore_case)))
  names(sel) <- names(strand_map)

  empty <- names(sel)[lengths(sel) == 0]
  if (length(empty))
    stop("no MS2 probe matches the pattern for: ", paste(empty, collapse = ", "),
         "\n  probes look like: ", paste(utils::head(ph, 6), collapse = ", "),
         "\n  adjust strand_map, e.g. strand_map = c('(+) strand' = '_rev', ",
         "'(-) strand' = '_fwd')")
  both <- if (length(sel) > 1) Reduce(intersect, sel) else character(0)
  if (length(both))
    stop(length(both), " probe(s) match BOTH strand patterns and would be ",
         "counted twice: ", paste(utils::head(both, 5), collapse = ", "))
  unassigned <- setdiff(ph, unlist(sel, use.names = FALSE))

  cat("\n--- MS2 probes by strand (label <- pattern; these are SWAPPED) ---\n")
  for (nm in names(sel))
    cat("  ", nm, "  <-  ", length(sel[[nm]]), " probes matching '",
        strand_map[[nm]], "'   (", paste(utils::head(sel[[nm]], 3), collapse = ", "),
        if (length(sel[[nm]]) > 3) ", ..." else "", ")\n", sep = "")
  if (length(unassigned))
    message("  ", length(unassigned), " phage probe(s) match neither pattern ",
            "and are NOT plotted: ",
            paste(utils::head(unassigned, 6), collapse = ", "),
            if (length(unassigned) > 6) ", ..." else "")

  cells <- qc$cell
  d <- data.table::rbindlist(lapply(names(sel), function(nm)
    data.table::data.table(
      cell   = cells,
      strand = nm,
      umi    = as.numeric(Matrix::colSums(cp[sel[[nm]], cells, drop = FALSE])))))
  d <- merge(d, qc[, .(cell, timepoint = as.numeric(timepoint),
                       condition = as.character(condition))], by = "cell")

  d[, strand := factor(strand, levels = names(strand_map))]
  d[, condition := factor(condition,
       levels = intersect(c("infected", "control"), unique(condition)))]
  tps <- sort(unique(d$timepoint))
  d[, tp := factor(sprintf("%g minutes", timepoint),
                   levels = sprintf("%g minutes", tps))]

  d[, xlog := log10(umi + 1)]
  hi <- max(d$xlog, na.rm = TRUE)
  if (!is.finite(hi) || hi <= 0) hi <- 1
  edges <- seq(0, hi, length.out = bins + 1)
  mids  <- (edges[-1] + edges[-length(edges)]) / 2
  bw    <- diff(edges)[1]

  h <- d[, {
    b  <- cut(xlog, breaks = edges, include.lowest = TRUE, labels = FALSE)
    tb <- tabulate(b, nbins = bins)
    .(mid = mids, frac = if (sum(tb)) tb / sum(tb) else tb)
  }, by = .(strand, tp, condition)]
  h <- h[frac > 0]

  top <- max(d$umi, na.rm = TRUE)
  brk <- c(0, 1, 10, 100, 1000, 1e4, 1e5)
  brk <- brk[brk <= max(top, 1)]
  if (length(brk) < 2) brk <- c(0, max(top, 1))

  lv   <- levels(h$condition)
  cols <- colours[lv]
  labs <- cond_labels[lv]

  fq <- list(rows = ggplot2::vars(strand), cols = ggplot2::vars(tp),
             switch = "y", scales = if (free_y) "free_y" else "fixed")
  facets <- if ("axes" %in% names(formals(ggplot2::facet_grid))) {
    do.call(ggplot2::facet_grid, c(fq, list(axes = "all_x")))
  } else if (requireNamespace("ggh4x", quietly = TRUE)) {
    do.call(ggh4x::facet_grid2, c(fq, list(axes = "all_x")))
  } else {
    message("  ggplot2 ", utils::packageVersion("ggplot2"),
            " has no `axes` argument and ggh4x is not installed -- the top row ",
            "gets no x axis. Upgrade ggplot2 to >= 3.5.0, or install ggh4x.")
    do.call(ggplot2::facet_grid, fq)
  }

  pl <- ggplot2::ggplot(h, ggplot2::aes(mid, frac, fill = condition)) +
    ggplot2::geom_col(width = bw, position = "identity",
                      alpha = alpha, colour = NA) +
    facets +
    ggplot2::scale_x_continuous(
      breaks = log10(brk + 1),
      labels = format(brk, big.mark = ",", scientific = FALSE, trim = TRUE),
      expand = ggplot2::expansion(mult = c(0.015, 0.03))) +
    ggplot2::scale_y_continuous(
      labels = function(z) sprintf("%.2f", z),
      expand = ggplot2::expansion(mult = c(0, 0.08))) +
    ggplot2::scale_fill_manual(values = cols, labels = labs, name = NULL) +
    ggplot2::labs(x = "MS2 UMIs per cell", y = "Fraction of cells") +
    paper_theme(base_size, "labels") +
    ggplot2::theme(
      legend.position   = "top",
      legend.margin     = ggplot2::margin(b = -4),
      legend.text       = ggplot2::element_text(size = base_size * 1.05),
      panel.spacing.x   = grid::unit(12, "pt"),
      panel.spacing.y   = grid::unit(16, "pt"),   # room for the top row's axis
      axis.title        = ggplot2::element_text(face = "plain",
                            size = base_size * 1.2),
      axis.text         = ggplot2::element_text(size = base_size * 1.1,
                            colour = "grey20"),
      axis.line         = ggplot2::element_blank(),
      axis.ticks        = ggplot2::element_line(colour = "grey20",
                            linewidth = 0.5),
      axis.ticks.length = grid::unit(4, "pt"),
      strip.placement   = "outside",
      strip.background  = ggplot2::element_blank(),
      strip.text.x      = ggplot2::element_text(face = "plain",
                            size = base_size * 1.2,
                            margin = ggplot2::margin(b = 5)),
      strip.text.y.left = ggplot2::element_text(face = "plain", angle = 90,
                            size = base_size * 1.1,
                            margin = ggplot2::margin(r = 5)))

  st <- d[, .(n = .N,
              pct_zero   = round(100 * mean(umi == 0), 1),
              median_umi = round(stats::median(umi), 1),
              mean_umi   = round(mean(umi), 1),
              max_umi    = max(umi)),
          by = .(strand, tp, condition)][order(strand, tp, condition)]
  cat("\nMS2 UMIs per cell (cells passing QC):\n"); print(st)

  n_row <- nlevels(d$strand); n_col <- nlevels(d$tp)
  width  <- width  %||% max(8, 1.4 + 4.2 * n_col)
  height <- height %||% (1.2 + 2.7 * n_row)
  name   <- name   %||% "SuppFig_MS2_UMI_by_strand"
  save_paper_fig(pl, name, width, height, 400, formats = formats, show = show)
}

## Run

In [ ]:
raw_exp1 <- stage_read(params_exp1)
invisible(diagnose_feature_names(rownames(raw_exp1$counts_probe), params_exp1))

## Figure S5

Pseudo-bulk of QC-passing cells of the 45-min infected library against bulk RNA-seq of the 45-min infected samples. Host genes with |standardized residual| > 3 are removed from the downstream single-cell analysis.

In [ ]:
counts_gene_exp1 <- stage_aggregate(raw_exp1, params_exp1, verbose = FALSE)
qc_exp1 <- stage_qc(raw_exp1, params_exp1, counts_gene = counts_gene_exp1)$qc
pb_cells <- qc_exp1[keep == TRUE & sample_id == "Inf_45m", cell]
pb_counts <- Matrix::rowSums(counts_gene_exp1[, pb_cells, drop = FALSE])

sc_df <- data.frame(panel_gene = names(pb_counts),
                    Gene = gsub("_", "-", names(pb_counts)),
                    pb_norm_inf = as.numeric(pb_counts) / sum(pb_counts) * 1e6)

bulk <- read.csv(BULK_COUNTS, row.names = 1, check.names = FALSE)
bulk <- as.matrix(bulk[!startsWith(rownames(bulk), "__"), c("L1M-45min", "L2M-45min", "L3M-45min")])
bulk_cpm <- sweep(bulk, 2, colSums(bulk), "/") * 1e6
bulk_df <- data.table(Gene = sub("_[0-9]$", "", rownames(bulk_cpm)), bulk_norm_inf = rowMeans(bulk_cpm))
bulk_df <- bulk_df[, .(bulk_norm_inf = mean(bulk_norm_inf)), by = Gene]
bulk_df[, Gene := sub("^LELOBEKK_", "LELOBEKK-", Gene)]

g <- sc_df$Gene
stripped <- sub("^LELOBEKK-", "", g)
sc_df$Gene <- ifelse(tolower(g) == "ms2-fwd", "MS2_negative",
              ifelse(tolower(g) == "ms2-rev", "MS2",
              ifelse(grepl("^LELOBEKK-", g) & g != "LELOBEKK-traY" & stripped %in% bulk_df$Gene,
                     stripped, g)))

cmp <- merge(sc_df, as.data.frame(bulk_df), by = "Gene")
cmp <- cmp[!is.na(cmp$pb_norm_inf) & !is.na(cmp$bulk_norm_inf), ]

fit <- lm(log10(bulk_norm_inf + 1) ~ log10(pb_norm_inf + 1), data = cmp)
cmp$Residual <- resid(fit)
cmp$Residual_Z <- scale(cmp$Residual)[, 1]
cmp$Is_Outlier <- abs(cmp$Residual_Z) > 3 & !(cmp$Gene %in% c("MS2", "MS2_negative"))

cor_stats <- cor.test(log10(cmp$pb_norm_inf + 1), log10(cmp$bulk_norm_inf + 1), method = "pearson")
cat("Pearson r =", round(cor_stats$estimate, 3), " R2 =", round(cor_stats$estimate^2, 3),
    " p =", signif(cor_stats$p.value, 3), "\n")

outlier_genes <- unique(probe_to_gene(cmp$panel_gene[cmp$Is_Outlier], params_exp1))
cat(length(outlier_genes), "outlier genes:", paste(sort(cmp$Gene[cmp$Is_Outlier]), collapse = ", "), "\n")
writeLines(outlier_genes, file.path(SC_DIR, "pseudobulk_outlier_genes.txt"))
params_exp1$outlier_genes <- outlier_genes

In [ ]:
r2_text <- paste0("italic(R)^2 == ", format(round(unname(cor_stats$estimate)^2, 3), nsmall = 3))
p_text  <- if (cor_stats$p.value < 1e-10) "italic(p) < 10^{-10}" else
  paste0("italic(p) == ", signif(cor_stats$p.value, 2))
anchor_x <- 10
anchor_y <- max(cmp$bulk_norm_inf + 1, na.rm = TRUE)

pS5 <- ggplot(cmp, aes(x = pb_norm_inf + 1, y = bulk_norm_inf + 1)) +
  geom_point(aes(color = Is_Outlier), alpha = 0.6, size = 2) +
  scale_color_manual(values = c("FALSE" = "dodgerblue4", "TRUE" = "darkred"),
                     labels = c("FALSE" = "Consistent", "TRUE" = "Outlier (|Z| > 3)")) +
  geom_abline(intercept = 0, slope = 1, color = "gray40", linewidth = 1.2) +
  annotate("text", x = 6.5e5, y = 3e5, label = "x = y", color = "black", fontface = "italic", size = 11) +
  geom_smooth(method = "lm", formula = y ~ x, color = "black", linetype = "dashed", se = FALSE, linewidth = 1) +
  scale_x_log10(breaks = 10^(0:6), labels = scales::trans_format("log10", scales::math_format(10^.x)),
                limits = c(1, NA), expand = expansion(mult = c(0, 0.05))) +
  scale_y_log10(breaks = 10^(0:6), labels = scales::trans_format("log10", scales::math_format(10^.x)),
                limits = c(1, NA), expand = expansion(mult = c(0, 0.05))) +
  labs(x = "Single cell pseudo bulk (CPM + 1)", y = "Bulk RNA-seq (CPM + 1)", color = "Expression Status") +
  theme_classic(base_size = 24) +
  theme(
    axis.title   = element_text(face = "plain", size = 28),
    axis.text    = element_text(size = 28, color = "black"),
    axis.line    = element_blank(),
    panel.border = element_rect(fill = NA, color = "black", linewidth = 0.8),
    axis.ticks   = element_line(linewidth = 0.8, color = "black"),
    axis.ticks.length = unit(0.3, "cm"),
    legend.title = element_text(face = "plain", size = 28),
    legend.text  = element_text(size = 28),
    legend.key.size = unit(1.5, "cm")
  ) +
  guides(color = guide_legend(override.aes = list(size = 8))) +
  geom_text_repel(data = cmp[cmp$Is_Outlier, ], aes(label = Gene), color = "darkred",
                  size = 8, fontface = "italic", max.overlaps = 30) +
  annotate("text", x = anchor_x, y = anchor_y, label = r2_text, parse = TRUE,
           hjust = 0, vjust = 1, size = 11, color = "black") +
  annotate("text", x = anchor_x, y = anchor_y / 3.5, label = p_text, parse = TRUE,
           hjust = 0, vjust = 1, size = 11, color = "black")

save_paper_fig(pS5, "FigS5", 18, 9, dpi = 300)
fwrite(cmp, file.path(SRC_DIR, "FigS5_data.csv"))

## Pipeline

In [ ]:
res_exp1 <- run_pipeline(params_exp1, cached_raw = raw_exp1)

In [ ]:
raw_inf180 <- stage_read(params_inf180)
res_inf180 <- run_pipeline(params_inf180, cached_raw = raw_inf180)

In [ ]:
REGULON_TPS <- c(15, 30, 45)

TC_SAMPLES <- read_sample_sheet(params_exp1$sample_sheet)[
  timepoint %in% REGULON_TPS, sample_id]

res_tc <- ensure_samples_read(res_exp1, TC_SAMPLES)

REGULONS <- lapply(names(REGULON_PATTERNS), function(m)
  panel_genes(res_tc, REGULON_PATTERNS[[m]],
              extra = REGULON_EXTRA[[m]] %||% character(0)))
names(REGULONS) <- names(REGULON_PATTERNS)
for (m in names(REGULONS))
  cat("  ", REGULON_PATTERNS[[m]], " -> ", length(REGULONS[[m]]), " gene(s): ",
      paste(REGULONS[[m]], collapse = ", "), "\n", sep = "")
if (any(lengths(REGULONS) == 0))
  stop("a pattern matched no genes on this panel: ",
       paste(names(REGULONS)[lengths(REGULONS) == 0], collapse = ", "))

cat("\nregulon genes on the probe panel:\n")
REGULON_FOUND <- lapply(names(REGULONS), function(m)
  regulon_present(res_tc, REGULONS[[m]], label = unname(REGULON_LABELS[m])))
names(REGULON_FOUND) <- names(REGULONS)

mod_tc <- regulon_module_test_pb(res_tc, REGULON_TPS, n_pools = 4)

mod_null <- regulon_module_null(res_tc, mod_tc, n_null = 2000)

cat("\n=== figure S9A source table ===\n")
print(mod_null[, .(timepoint, module, n_genes,
                   n_exposed = n_bystander, n_control,
                   rate_control   = round(rate_control, 2),
                   rate_exposed   = round(rate_bystander, 2),
                   fold = round(fold, 2),
                   ci = paste0(round(2^ci_lo, 2), "-", round(2^ci_hi, 2)),
                   null_fold = round(2^null_mean, 2),
                   pct = round(percentile, 1), stars)])

fwrite(mod_null, file.path(SRC_DIR, "FigS9A_data.csv"))

## Figures

In [ ]:
MS2_SPECS <- list(list(res = res_exp1, sample = "Inf_45m"))
if (!is.null(res_inf180))
  MS2_SPECS <- c(MS2_SPECS, list(list(res = res_inf180, sample = "INF_180min")))

MS2_LINEAR_LIMITS <- shared_ms2_limits(MS2_SPECS, clip = c(0.01, 0.99),
                                       round_to = 5)

MS2_LIMITS <- c(0.5, 99)
MS2_BREAKS <- c(1, 5, 20, 50, 80, 95, 99)
MS2_TRANS  <- logit_percent_trans()

MS2_COLOURS_DARK <- c("#0B3D6B", "#2E6DA4", "#6E8CA0", "#A88A63",
                      "#E08A2E", "#B34A0C")

MS2_COLOURS <- c("#08306B", "#2171B5", "#6BAED6", "#BDD7E7",
                 "#FDD0A2", "#FD8D3C", "#E6550D", "#A63603")

cat("\ncolour scale: logit, ", MS2_LIMITS[1], "% to ", MS2_LIMITS[2],
    "%  (linear equivalent would have been ", MS2_LINEAR_LIMITS[1], "-",
    MS2_LINEAR_LIMITS[2], "%)\n", sep = "")

### Figure 2A

In [ ]:
fig2A <- fig_clusters_ms2(res_exp1, "Inf_45m", ms2_limits = MS2_LIMITS,
                          ms2_trans = MS2_TRANS, ms2_breaks = MS2_BREAKS,
                          palette = MS2_COLOURS_DARK,
                          fig_title = "45 minutes", name = "Fig2A")
fwrite(umap_table(res_exp1, "Inf_45m"), file.path(SRC_DIR, "Fig2A_data.csv"))

### Figure 2B

In [ ]:
PLASMID_SAMPLES <- c("Inf_30m", "Inf_45m")
F_GMAP <- .resolve_f_genes(res_tc, F_TRANSFER_MAP)
fig2B <- fig_plasmid_transfer(res_tc, PLASMID_SAMPLES, gmap = F_GMAP, name = "Fig2B")
fwrite(fig2B$module, file.path(SRC_DIR, "Fig2B_module_data.csv"))
fwrite(fig2B$genes, file.path(SRC_DIR, "Fig2B_gene_data.csv"))

### Figure 2C

In [ ]:
fig2C <- fig_ms2_only(res_inf180, "INF_180min", ms2_limits = MS2_LIMITS,
                      ms2_trans = MS2_TRANS, ms2_breaks = MS2_BREAKS,
                      palette = MS2_COLOURS_DARK, panel_title = "180 minutes", name = "Fig2C")
fwrite(umap_table(res_inf180, "INF_180min"), file.path(SRC_DIR, "Fig2C_data.csv"))

### Figure S6

Probes named `rev` read the (+) strand and those named `fwd` the (−) strand.

In [ ]:
figS6 <- fig_ms2_strand_hist(res_tc, samples = TC_SAMPLES,
                             strand_map = c("(+) strand" = "rev", "(-) strand" = "fwd"),
                             formats = FIG_FORMATS, name = "FigS6")

### Figure S7

Sensitivity of the infection calls to the lower host QC cutoffs of the 30- and 45-min infected libraries. The control libraries are QC'd with the standard parameters and define the infection cutoffs, which stay fixed. For the grid the lower quantile cutoffs are switched off and the listed minimum host UMIs and genes are applied directly; the upper cutoffs are unchanged.

In [ ]:
QC_GRID_SAMPLES <- c("Inf_30m", "Inf_45m")
QC_GRID_UMI     <- c(100, 75, 50, 25, 10)
QC_GRID_GENES   <- c(50, 40, 30)

stage_qc_metrics <- function(raw, p, counts_gene = NULL) {
  if (is.null(counts_gene)) counts_gene <- stage_aggregate(raw, p, verbose = FALSE)
  feat_basis  <- p$qc_feat_basis  %||% p$qc_basis %||% "probe"
  umi_basis   <- p$qc_umi_basis   %||% p$qc_basis %||% "probe"
  phage_basis <- p$qc_phage_basis %||% "probe"
  qc <- compute_qc_metrics(
          .qc_basis_matrix(umi_basis, raw, counts_gene, "qc_umi_basis"),
          raw$meta, p,
          mat_feat  = .qc_basis_matrix(feat_basis,  raw, counts_gene, "qc_feat_basis"),
          mat_phage = .qc_basis_matrix(phage_basis, raw, counts_gene, "qc_phage_basis"))
  vvp <- compute_valid_viral_probes(raw$counts_probe, p)
  setkey(vvp, cell); setkey(qc, cell)
  qc <- vvp[qc]
  if (anyNA(qc$valid_viral_probes))
    stop(sum(is.na(qc$valid_viral_probes)), " cells have NA valid_viral_probes")
  qc[]
}

grid_qc_params <- function(p, min_umi, min_genes)
  modifyList(p, list(min_host_umi = min_umi, min_host_features = min_genes,
                     umi_lower_q = 0, feat_lower_q = 0))

summarize_calls <- function(qc_cls, samples, setting) {
  qc_cls[sample_id %in% samples,
         .(setting        = setting,
           min_host_umi   = round(umi_lo[1], 1),
           min_host_genes = round(feat_lo[1], 1),
           n_cells        = .N,
           n_infected     = sum(infection == "infected"),
           pct_infected   = 100 * mean(infection == "infected")),
         by = sample_id]
}

In [ ]:
params_grid <- modifyList(params_exp1, list(
  analysis_samples = c("Ctrl_15m", "Ctrl_30m", "Ctrl_45m", QC_GRID_SAMPLES)))
raw_grid <- stage_read(params_grid)
counts_gene_grid <- stage_aggregate(raw_grid, params_grid, verbose = FALSE)
qc_metrics <- stage_qc_metrics(raw_grid, params_grid, counts_gene = counts_gene_grid)

thr_base  <- compute_qc_thresholds(qc_metrics, params_grid)
qc_base   <- apply_qc_filter(qc_metrics, thr_base, params_grid)
cls_base  <- classify_infection(qc_base$qc[keep == TRUE], params_grid)
ctrl_kept <- qc_base$qc[keep == TRUE & condition == "control"]

qc_inf     <- qc_metrics[sample_id %in% QC_GRID_SAMPLES]
upper_base <- thr_base[sample_id %in% QC_GRID_SAMPLES][order(sample_id), .(umi_hi, feat_hi)]
grid <- CJ(min_host_umi = QC_GRID_UMI, min_host_genes = QC_GRID_GENES, sorted = FALSE)

grid_tab <- rbindlist(lapply(seq_len(nrow(grid)), function(i) {
  p_i   <- grid_qc_params(params_grid, grid$min_host_umi[i], grid$min_host_genes[i])
  thr_i <- compute_qc_thresholds(qc_inf, p_i)
  if (!isTRUE(all.equal(thr_i[order(sample_id), .(umi_hi, feat_hi)], upper_base)))
    stop("upper QC cutoffs moved at grid point ", i)
  qc_i  <- apply_qc_filter(qc_inf, thr_i, p_i)$qc[keep == TRUE]
  cls_i <- classify_infection(rbind(ctrl_kept, qc_i, use.names = TRUE, fill = TRUE),
                              p_i, verbose = FALSE)
  if (!isTRUE(all.equal(cls_i$cutoffs[, .(thr_ambiguous, thr_infected)],
                        cls_base$cutoffs[, .(thr_ambiguous, thr_infected)])))
    stop("infection cutoffs moved at grid point ", i)
  summarize_calls(cls_i$qc, QC_GRID_SAMPLES, "grid")
}))
print(rbind(summarize_calls(cls_base$qc, QC_GRID_SAMPLES, "current"), grid_tab), nrows = Inf)
fwrite(grid_tab, file.path(SRC_DIR, "FigS7_data.csv"))

In [ ]:
d <- copy(grid_tab)
d[, panel := factor(paste(sub("Inf_([0-9]+)m", "\\1", sample_id), "minutes"),
                    levels = paste(sub("Inf_([0-9]+)m", "\\1", QC_GRID_SAMPLES), "minutes"))]
d[, umi   := factor(min_host_umi, levels = sort(QC_GRID_UMI))]
d[, genes := factor(min_host_genes, levels = sort(QC_GRID_GENES))]
d[, label := sprintf("%.1f%%\n(n = %s)", pct_infected, formatC(n_cells, format = "d", big.mark = ","))]
fill_lo <- 5 * floor(min(d$pct_infected) / 5)

pS7 <- ggplot(d, aes(genes, umi, fill = pct_infected)) +
  geom_tile(colour = "white", linewidth = 0.8) +
  geom_text(aes(label = label), size = 5) +
  facet_wrap(~ panel, nrow = 1) +
  scale_fill_gradient(low = "#FFF5EB", high = "#FF7F0E", limits = c(fill_lo, 100),
                      name = "Infected\ncells (%)\n") +
  labs(x = "Minimum host genes", y = "Minimum host UMIs") +
  theme_minimal(base_size = 20) +
  theme(panel.grid = element_blank(),
        strip.text = element_text(size = 24),
        axis.text  = element_text(colour = "black", size = 20),
        axis.title = element_text(size = 24),
        axis.title.x = element_text(margin = margin(t = 12)),
        axis.title.y = element_text(margin = margin(r = 12)),
        legend.title = element_text(size = 20),
        legend.text  = element_text(size = 18),
        legend.key.height = unit(1.4, "cm"))

save_paper_fig(pS7, "FigS7", 14, 8, 400)

### Figure S8

In [ ]:
figS8 <- fig_operon_grid(res_exp1, samples = c("Ctrl_45m", "Inf_45m"),
                         operons = list("fim operon" = "^fim", "srl operon" = "^srl"),
                         name = "FigS8")

### Figure S9A

In [ ]:
figS9A <- fig_regulon_timecourse(res_tc, REGULON_TPS, mod_stats = mod_null,
                                 bar_label = "fold", min_lfc = 1, name = "FigS9A")

### Figure S9B

In [ ]:
rates <- regulon_rate_table(res_tc, REGULON_TPS, groups = c("exposed", "control", "infected"))
fwrite(rates, file.path(SRC_DIR, "FigS9B_data.csv"))
figS9B <- fig_regulon_rates(res_tc, REGULON_TPS, rates = rates,
                           groups = c("exposed", "control", "infected"), name = "FigS9B")